# Data Extraction Module for Pricing & Offers System


## Imports


In [1]:
%%capture

# Upgrade pip
!pip install --upgrade pip
# Connectivity
!pip install psycopg2-binary

!pip install snowflake-connector-python==3.15.0
!pip install snowflake-sqlalchemy
!pip install warnings
!pip install keyring==23.11.0
!pip install sqlalchemy==1.4.46
!pip install requests
!pip install boto3
!pip install oauth2client
!pip install gspread==5.9.0
!pip install gspread_dataframe
!pip install google.cloud
# Data manipulation and analysis
!pip install polars
!pip install pandas==2.2.1
!pip install numpy
!pip install openpyxl
!pip install xlsxwriter
# Date and time handling
!pip install --upgrade datetime
!pip install python-time
!pip install --upgrade pytz
# Progress bar
!pip install tqdm
# Database data types
!pip install db-dtypes
# Modeling
!pip install statsmodels
!pip install scikit-learn
!pip install import-ipynb
# Plotting
!pip install matplotlib
!pip install seaborn
!pip install aiohttp
#slack
!pip install slack_sdk
!pip install slack

In [2]:
import os
import numpy as np
import pandas as pd
import snowflake.connector
import setup_environment_2
import slack
import pytz
from common_functions import upload_dataframe_to_snowflake,send_text_slack

# Cairo timezone for consistent timestamps
CAIRO_TZ = pytz.timezone('Africa/Cairo')


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


## Variables


In [3]:
# Region
REGION = "Egypt"

# Snowflake Warehouse
WAREHOUSE = "COMPUTE_WH"

# Date Variables
from datetime import datetime, timedelta
TODAY = datetime.now(CAIRO_TZ).date()
YESTERDAY = TODAY - timedelta(days=1)


## Initialize Environment


In [4]:
setup_environment_2.initialize_env()


/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


## Warehouse & Cohort Mapping


In [5]:
from constants import WAREHOUSE_MAPPING, COHORT_IDS


## Snowflake Query Function


In [6]:
from db import query_snowflake, get_snowflake_timezone


In [7]:
# get_snowflake_timezone is imported from db in the cell above.


## Helper Functions


In [8]:
def get_warehouse_df():
    """Get warehouse mapping as DataFrame."""
    return pd.DataFrame(
        WAREHOUSE_MAPPING,
        columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
    


def convert_to_numeric(df):
    """Convert DataFrame columns to numeric where possible."""
    df.columns = df.columns.str.lower()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
    return df


## Get Snowflake Timezone


In [9]:
TIMEZONE = get_snowflake_timezone()
print(f"Snowflake timezone: {TIMEZONE}")


Snowflake timezone: America/Los_Angeles


In [10]:
# =============================================================================
# 8. ALL-TIME HIGH MARGIN QUERY (IQR-filtered max margin over 240-day window)
# Uses a 30-day moving IQR to remove outlier margin days, then takes the max
# =============================================================================
ALL_TIME_HIGH_MARGIN_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
daily_margin_data AS (
    SELECT 
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        so.created_at::DATE AS sale_date,
        SUM(pso.total_price) AS daily_nmv,
        SUM(COALESCE(f.wac_p, 0) * pso.purchased_item_count * pso.basic_unit_count) AS daily_cogs,
        CASE 
            WHEN SUM(pso.total_price) > 0 
            THEN (SUM(pso.total_price) - SUM(COALESCE(f.wac_p, 0) * pso.purchased_item_count * pso.basic_unit_count)) / SUM(pso.total_price)
            ELSE 0 
        END AS daily_margin
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    JOIN finance.all_cogs f ON f.product_id = pso.product_id
        AND f.from_date::DATE <= so.created_at::DATE 
        AND f.to_date::DATE > so.created_at::DATE
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 240
        AND so.created_at::DATE < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), pso.product_id, so.created_at::DATE
),
moving_stats AS (
    SELECT 
        curr.warehouse_id,
        curr.product_id,
        curr.sale_date,
        curr.daily_margin,
        curr.daily_nmv,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY hist.daily_margin) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY hist.daily_margin) AS q3
    FROM daily_margin_data curr
    JOIN daily_margin_data hist 
      ON curr.warehouse_id = hist.warehouse_id 
      AND curr.product_id = hist.product_id
      AND hist.sale_date BETWEEN DATEADD(day, -30, curr.sale_date) AND curr.sale_date
    GROUP BY ALL
)
SELECT 
    product_id,
    warehouse_id,
    CASE WHEN max_margin < 0 THEN 0.07 ELSE max_margin END AS all_time_high_margin
FROM (
    SELECT 
        product_id,
        warehouse_id,
        MAX(daily_margin) AS max_margin
    FROM (
        SELECT *,
            (q3 - q1) AS iqr,
            (q1 - 1.5 * (q3 - q1)) AS lower_bound,
            (q3 + 1.5 * (q3 - q1)) AS upper_bound
        FROM moving_stats
        WHERE daily_margin BETWEEN (q1 - 1.5 * (q3 - q1)) AND (q3 + 1.5 * (q3 - q1))
        GROUP BY ALL
    )
    GROUP BY ALL
)
'''

print("All-time high margin query defined (IQR-filtered max margin)")


All-time high margin query defined (IQR-filtered max margin)


## Market Prices Extraction Queries
Queries for external market price data:
1. **Ben Soliman Prices** - Competitor reference prices
2. **Marketplace Prices** - Min, Max, Mod prices from marketplace
3. **Scrapped Data** - Competitor prices from scraping


In [11]:
## Additional Data Queries (Sales, Groups, WAC)


In [12]:
# =============================================================================
# 4. PRODUCT BASE DATA QUERY (product_id, sku, brand, cat, wac1, wac_p, current_price)
# =============================================================================
PRODUCT_BASE_QUERY = f'''
WITH skus_prices AS (
    WITH local_prices AS (
        SELECT  
            CASE 
                WHEN cpu.cohort_id IN (700, 695) THEN 'Cairo'
                WHEN cpu.cohort_id IN (701) THEN 'Giza'
                WHEN cpu.cohort_id IN (704, 698) THEN 'Delta East'
                WHEN cpu.cohort_id IN (703, 697) THEN 'Delta West'
                WHEN cpu.cohort_id IN (696, 1123, 1124, 1125, 1126) THEN 'Upper Egypt'
                WHEN cpu.cohort_id IN (702, 699) THEN 'Alexandria'
            END AS region,
            cohort_id,
            pu.product_id,
            pu.packing_unit_id,
            pu.basic_unit_count,
            AVG(cpu.price) AS price
        FROM cohort_product_packing_units cpu
        JOIN PACKING_UNIT_PRODUCTS pu ON pu.id = cpu.product_packing_unit_id
        WHERE cpu.cohort_id IN (700,701,702,703,704,695,696,697,698,699,1123,1124,1125,1126)
            AND cpu.created_at::date <> '2023-07-31'
            AND cpu.is_customized = TRUE
        GROUP BY ALL
    ),
    
    live_prices AS (
        SELECT 
            region, cohort_id, product_id, 
            pu_id AS packing_unit_id, 
            buc AS basic_unit_count, 
            NEW_PRICE AS price
        FROM materialized_views.DBDP_PRICES
        WHERE created_at = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date
            AND DATE_PART('hour', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::time) 
                BETWEEN SPLIT_PART(time_slot, '-', 1)::int AND (SPLIT_PART(time_slot, '-', 1)::int) + 1
            AND cohort_id IN (700,701,702,703,704,695,696,697,698,699,1123,1124,1125,1126)
    ),
    
    prices AS (
        SELECT *
        FROM (
            SELECT *, 1 AS priority FROM live_prices
            UNION ALL
            SELECT *, 2 AS priority FROM local_prices
        )
        QUALIFY ROW_NUMBER() OVER (PARTITION BY region, cohort_id, product_id, packing_unit_id ORDER BY priority) = 1
    )
    
    SELECT region, cohort_id, product_id, price
    FROM prices
    WHERE basic_unit_count = 1
        AND ((product_id = 1309 AND packing_unit_id = 2) OR (product_id <> 1309))
)

SELECT distinct
    region, cohort_id, p.product_id,
    CONCAT(products.name_ar, ' ', products.size, ' ', product_units.name_ar) AS sku,
    b.name_ar AS brand,
    cat.name_ar AS cat,
    wac1, wac_p, p.price as current_price
FROM skus_prices p
JOIN finance.all_cogs c ON c.product_id = p.product_id 
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN c.from_date AND c.to_date
JOIN products ON products.id = p.product_id
JOIN categories cat ON cat.id = products.category_id
JOIN brands b ON b.id = products.brand_id
JOIN product_units ON product_units.id = products.unit_id
WHERE wac1 > 0 AND wac_p > 0
GROUP BY ALL
'''

# =============================================================================
# 5. SALES DATA QUERY (120-day NMV by cohort/product)
# =============================================================================
SALES_QUERY = f'''
SELECT DISTINCT cpc.cohort_id, pso.product_id,
    CONCAT(products.name_ar,' ',products.size,' ',product_units.name_ar) as sku,
    brands.name_ar as brand, categories.name_ar as cat,
    sum(pso.total_price) as nmv
FROM product_sales_order pso
JOIN sales_orders so ON so.id = pso.sales_order_id
JOIN COHORT_PRICING_CHANGES cpc ON cpc.id = pso.COHORT_PRICING_CHANGE_id
JOIN products ON products.id = pso.product_id
JOIN brands ON products.brand_id = brands.id 
JOIN categories ON products.category_id = categories.id
JOIN product_units ON product_units.id = products.unit_id 
WHERE so.created_at::date BETWEEN CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 120 
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 1 
    AND so.sales_order_status_id NOT IN (7, 12)
    AND so.channel IN ('telesales', 'retailer')
    AND pso.purchased_item_count <> 0
    AND cpc.cohort_id IN (700,701,702,703,704,1123,1124,1125,1126)
GROUP BY ALL
'''

# =============================================================================
# 6. MARGIN STATS QUERY (STD and average margins)  
# =============================================================================
MARGIN_STATS_QUERY = f'''
select product_id, cohort_id, 
    (0.6*product_std) + (0.3*brand_std) + (0.1*cat_std) as std, 
    avg_margin
from (
    select product_id, cohort_id, 
        stddev(product_margin) as product_std, 
        stddev(brand_margin) as brand_std,
        stddev(cat_margin) as cat_std, 
        avg(product_margin) as avg_margin
    from (
        select distinct product_id, order_date, cohort_id,
            (nmv-cogs_p)/nullif(nmv,0) as product_margin, 
            (brand_nmv-brand_cogs)/nullif(brand_nmv,0) as brand_margin,
            (cat_nmv-cat_cogs)/nullif(cat_nmv,0) as cat_margin
        from (
            SELECT DISTINCT so.created_at::date as order_date, cpc.cohort_id, pso.product_id,
                brands.name_ar as brand, categories.name_ar as cat,
                sum(COALESCE(f.wac_p,0) * pso.purchased_item_count * pso.basic_unit_count) as cogs_p,
                sum(pso.total_price) as nmv,
                sum(nmv) over(partition by order_date, cat, brand) as brand_nmv,
                sum(cogs_p) over(partition by order_date, cat, brand) as brand_cogs,
                sum(nmv) over(partition by order_date, cat) as cat_nmv,
                sum(cogs_p) over(partition by order_date, cat) as cat_cogs
            FROM product_sales_order pso
            JOIN sales_orders so ON so.id = pso.sales_order_id   
            JOIN COHORT_PRICING_CHANGES cpc on cpc.id = pso.cohort_pricing_change_id
            JOIN products on products.id = pso.product_id
            JOIN brands on products.brand_id = brands.id 
            JOIN categories ON products.category_id = categories.id
            JOIN finance.all_cogs f ON f.product_id = pso.product_id
                AND f.from_date::date <= so.created_at::date AND f.to_date::date > so.created_at::date
            WHERE so.created_at::date between 
                date_trunc('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 120) 
                and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date
                AND so.sales_order_status_id not in (7,12)
                AND so.channel IN ('telesales','retailer')
                AND pso.purchased_item_count <> 0 
            GROUP BY ALL
            
        )
        where nmv > 0
    ) group by all 
)
'''

# =============================================================================
# 7. TARGET MARGINS QUERY
# =============================================================================
TARGET_MARGINS_QUERY = f'''
WITH cat_brand_target as (
    SELECT DISTINCT cat, brand, margin as target_bm
    FROM performance.commercial_targets cplan
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
),
cat_target as (
    select cat, sum(target_bm * (target_nmv/cat_total)) as cat_target_margin
    from (
        select *, sum(target_nmv) over(partition by cat) as cat_total
        from (
            select cat, brand, avg(target_bm) as target_bm, sum(target_nmv) as target_nmv
            from (
                SELECT DISTINCT date, city as region, cat, brand, margin as target_bm, nmv as target_nmv
                FROM performance.commercial_targets cplan
                QUALIFY CASE 
                    WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date) 
                    THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date)
                    ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - INTERVAL '1 month') 
                END = DATE_TRUNC('month', date)
            ) group by all
        )
    ) group by all 
)
SELECT DISTINCT cbt.cat, cbt.brand, cbt.target_bm, ct.cat_target_margin
FROM cat_brand_target cbt
LEFT JOIN cat_target ct ON ct.cat = cbt.cat
'''


In [13]:
## Execute All Queries

In [14]:
# =============================================================================
# Execute all queries
# =============================================================================
print("Loading data from Snowflake...")

# NOTE: Ben Soliman, Marketplace, and Scrapped prices are now fetched via
# market_data_module.ipynb get_market_data() function - no need to load here

# 1. Product Base Data (product_id, sku, brand, cat, wac1, wac_p, current_price)
print("  1. Loading product base data...")
df_product_base = query_snowflake(PRODUCT_BASE_QUERY)
df_product_base = convert_to_numeric(df_product_base)
print(f"     Loaded {len(df_product_base)} product base records")

# 2. Sales Data
print("  2. Loading sales data...")
df_sales = query_snowflake(SALES_QUERY)
df_sales = convert_to_numeric(df_sales)
print(f"     Loaded {len(df_sales)} sales records")

# 3. Margin Stats
print("  3. Loading margin stats...")
df_margin_stats = query_snowflake(MARGIN_STATS_QUERY)
df_margin_stats = convert_to_numeric(df_margin_stats)
print(f"     Loaded {len(df_margin_stats)} margin stat records")

# 4. Target Margins
print("  4. Loading target margins...")
df_targets = query_snowflake(TARGET_MARGINS_QUERY)
df_targets = convert_to_numeric(df_targets)
print(f"     Loaded {len(df_targets)} target margin records")

# 5. Product Groups (from PostgreSQL)
print("  5. Loading product groups...")
df_groups = query_snowflake(
    '''SELECT * FROM materialized_views.sku_commercial_groups'''
)
df_groups.columns = df_groups.columns.str.lower()
df_groups = convert_to_numeric(df_groups)
print(f"     Loaded {len(df_groups)} group records")

# 6. All-Time High Margin (P80 margin weighted by gross profit)
print("  6. Loading all-time high margin data...")
df_all_time_high_margin = query_snowflake(ALL_TIME_HIGH_MARGIN_QUERY)
df_all_time_high_margin = convert_to_numeric(df_all_time_high_margin)
print(f"     Loaded {len(df_all_time_high_margin)} all-time high margin records")

print("\nBase queries completed!")
print("NOTE: Market data (Ben Soliman, Marketplace, Scrapped) will be fetched via market_data_module")
print(f"\n{'='*60}")
print("df_product_base DataFrame available with columns:")
print("  - region, cohort_id, product_id, sku, brand, cat, wac1, wac_p, current_price")
print(f"{'='*60}")


Loading data from Snowflake...
  1. Loading product base data...


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


     Loaded 104759 product base records
  2. Loading sales data...


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


     Loaded 21968 sales records
  3. Loading margin stats...


...

/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


     Loaded 484 target margin records
  5. Loading product groups...


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


     Loaded 1938 group records
  6. Loading all-time high margin data...
     Loaded 35627 all-time high margin records

Base queries completed!
NOTE: Market data (Ben Soliman, Marketplace, Scrapped) will be fetched via market_data_module

df_product_base DataFrame available with columns:
  - region, cohort_id, product_id, sku, brand, cat, wac1, wac_p, current_price


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [20]:
# =============================================================================
# PART A: Get market_data from market_data_module
# =============================================================================
# Instead of duplicating the market data processing logic here, 
# we use the centralized market_data_module which handles:
# - Ben Soliman prices
# - Marketplace prices  
# - Scrapped prices
# - Group-level price aggregation
# - Price coverage filtering
# - Price percentile calculation
# - Margin tier conversion
# =============================================================================

print("Loading market_data from market_data_module...")
print("(This fetches Ben Soliman, Marketplace, and Scrapped prices from Snowflake)")
print()

# Run market_data_module_2 (loads V1 internally for legacy + adds V2 functions)
%run modules/market_data_module_2.ipynb

# Get legacy market data for DB storage (same output as V1)
market_data = get_market_data_legacy()

# The market_data now contains:
# - cohort_id, product_id, region
# - Raw prices: ben_soliman_price, final_min_price, final_max_price, etc.
# - Percentiles: minimum, percentile_25, percentile_50, percentile_75, maximum
# - Margins: below_market, market_min, market_25, market_50, market_75, market_max, above_market

print(f"\n{'='*60}")
print(f"MARKET DATA LOADED FROM MODULE")
print(f"{'='*60}")
print(f"Total records: {len(market_data)}")
print(f"  - With market data: {(~market_data['market_min'].isna()).sum()}")
print(f"  - Market data source: {market_data['market_data_source'].value_counts(dropna=False).to_dict()}")


Loading market_data from market_data_module...
(This fetches Ben Soliman, Marketplace, and Scrapped prices from Snowflake)

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  

## PART A: Build Main pricing_data DataFrame
Start with df_product_base (all our SKUs) and LEFT JOIN the processed market_data


In [21]:
# =============================================================================
# PART B: Build Main pricing_data DataFrame from df_product_base
# =============================================================================
print("Building main pricing_data DataFrame...")

# =============================================================================
# Step 1: Start with df_product_base as the MAIN dataframe (all our SKUs)
# =============================================================================
print("  Step 1: Starting with product base (all SKUs)...")
pricing_data = df_product_base.copy()
print(f"     Product base: {len(pricing_data)} records")

# =============================================================================
# Step 2: Add warehouse mapping (warehouse_id and warehouse name)
# =============================================================================
print("  Step 2: Adding warehouse mapping...")
warehouse_df = get_warehouse_df()
pricing_data = pricing_data.merge(
    warehouse_df[['cohort_id', 'warehouse_id', 'warehouse']], 
    on='cohort_id'
)
print(f"     After warehouse mapping: {len(pricing_data)} records")

# =============================================================================
# Step 3: LEFT JOIN processed market_data
# =============================================================================
print("  Step 3: Left joining processed market data...")
pricing_data = pricing_data.merge(
    market_data, 
    on=['cohort_id', 'product_id','region'], 
    how='left'
)
print(f"     After market data join: {len(pricing_data)} records")

print(f"     Market data source distribution:")
print(f"       {pricing_data['market_data_source'].value_counts(dropna=False).to_dict()}")

# =============================================================================
# Step 4: LEFT JOIN supporting data (sales, margins, targets, groups)
# =============================================================================
print("  Step 4: Left joining supporting data...")

# Merge sales data (nmv only)
pricing_data = pricing_data.merge(
    df_sales[['cohort_id', 'product_id', 'nmv']], 
    on=['cohort_id', 'product_id'], 
    how='left'
)
pricing_data['nmv'] = pricing_data['nmv'].fillna(0)

# Merge margin statistics (by cohort_id + product_id)
pricing_data = pricing_data.merge(df_margin_stats, on=['cohort_id', 'product_id'], how='left')

# Merge target margins (by brand + cat)
pricing_data = pricing_data.merge(df_targets, on=['brand', 'cat'], how='left')
pricing_data['target_margin'] = pricing_data['target_bm'].fillna(pricing_data['cat_target_margin']).fillna(0)
pricing_data = pricing_data.drop(columns=['target_bm', 'cat_target_margin'], errors='ignore')

# Fill NaN values with defaults
pricing_data['std'] = pricing_data['std'].fillna(0.01)
pricing_data['avg_margin'] = pricing_data['avg_margin'].fillna(0)

# Merge product groups
pricing_data = pricing_data.merge(df_groups, on='product_id', how='left')

# =============================================================================
# Step 5: Calculate current margin
# =============================================================================
pricing_data['current_margin'] = (pricing_data['current_price'] - pricing_data['wac_p']) / pricing_data['current_price']

# Remove duplicates
pricing_data = pricing_data.drop_duplicates(subset=['cohort_id', 'product_id','warehouse_id'])

# =============================================================================
# Reorder columns
# =============================================================================
final_columns = [
    # Product Base Info
    'cohort_id', 'product_id', 'region', 'warehouse_id', 'warehouse', 'sku', 'brand', 'cat',
    # Cost & Price
    'wac1', 'wac_p', 'current_price', 'current_margin',
    # Sales
    'nmv',
    # Market Prices (raw)
    'ben_soliman_price', 
    'final_min_price', 'final_max_price', 'final_mod_price', 'final_true_min', 'final_true_max',
    'min_scrapped', 'scrapped25', 'scrapped50', 'scrapped75', 'max_scrapped',
    # Price Percentiles
    'minimum', 'percentile_25', 'percentile_50', 'percentile_75', 'maximum',
    # Margin Tiers
    'below_market', 'market_min', 'market_25', 'market_50', 'market_75', 'market_max', 'above_market',
    # Supporting Data
    'std', 'avg_margin', 'target_margin', 'group'
]
pricing_data = pricing_data[[c for c in final_columns if c in pricing_data.columns]]

print(f"\n{'='*60}")
print(f"PRICING DATA COMPLETE")
print(f"{'='*60}")
print(f"Total records: {len(pricing_data)}")
print(f"\nRecords with market data: {len(pricing_data[~pricing_data['minimum'].isna()])}")
print(f"Records without market data: {len(pricing_data[pricing_data['minimum'].isna()])}")
print(f"\nRecords with sales (nmv > 0): {len(pricing_data[pricing_data['nmv'] > 0])}")
print(f"Records without sales (nmv = 0): {len(pricing_data[pricing_data['nmv'] == 0])}")
print(f"\nSample data:")
pricing_data.head()


Building main pricing_data DataFrame...
  Step 1: Starting with product base (all SKUs)...
     Product base: 104759 records
  Step 2: Adding warehouse mapping...
     After warehouse mapping: 88634 records
  Step 3: Left joining processed market data...
     After market data join: 88634 records
     Market data source distribution:
       {'sku': 33144, nan: 31390, 'brand': 24100}
  Step 4: Left joining supporting data...

PRICING DATA COMPLETE
Total records: 88626

Records with market data: 57244
Records without market data: 31382

Records with sales (nmv > 0): 31000
Records without sales (nmv = 0): 57626

Sample data:


,cohort_id,product_id,region,warehouse_id,warehouse,sku,brand,cat,wac1,wac_p,...,below_market,market_min,market_25,market_50,market_75,market_max,above_market,std,avg_margin,target_margin
0,704,5811,Delta East,339,Mansoura FC,بسكويت بيمبو بندق - 5 جنية,بيمبو,بسكويت و معمول,47.500000,47.185177,...,0.022069,48.25,49.1875,50.375,51.625,54.25,0.130227,0.025721,0.073629,0.070000
1,704,5811,Delta East,170,Sharqya,بسكويت بيمبو بندق - 5 جنية,بيمبو,بسكويت و معمول,47.500000,47.185177,...,0.022069,48.25,49.1875,50.375,51.625,54.25,0.130227,0.025721,0.073629,0.070000
2,700,10594,Cairo,1,Mostorod,كورونا شوكولاتة داكنة 72%- 45 جنية,كورونا,شوكولاتة,228.187858,226.401841,...,0.040670,236.00,246.4375,258.625,274.875,296.25,0.235774,0.023335,0.056982,0.070000
3,1124,6224,Upper Egypt,501,Assiut FC,بانيه كوكى توابل شرقية بارد 20 قطعة - 1 كجم,كوكى,مجمدات دجاج,126.883321,117.130886,...,0.049648,123.25,130.0000,137.250,146.125,156.25,0.250362,0.010000,0.000000,0.050000
5,1126,24709,Upper Egypt,401,Bani sweif,بلـيزو سوداني محشو بالشوكولاتة مغطي بالسوداني ...,نوفي,ويفر,44.180000,44.180000,...,0.084352,48.25,49.7500,51.000,52.500,54.75,0.193059,0.010000,0.000000,0.066787


## Discount Analysis - Price & Margin After Discount


In [23]:
# =============================================================================
# Discount Query - Get discount percentage by warehouse and product
# =============================================================================
DISCOUNT_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id, total_discount/total_nmv AS discount_perc
FROM (
    SELECT  
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        SUM(pso.total_price) AS total_nmv,
        SUM((ITEM_QUANTITY_DISCOUNT_VALUE * pso.purchased_item_count) + 
            (ITEM_DISCOUNT_VALUE * pso.purchased_item_count)) AS total_discount
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY ALL
)
WHERE total_nmv > 0
'''

# Execute discount query
print("Loading discount data...")
df_discount = query_snowflake(DISCOUNT_QUERY)
df_discount = convert_to_numeric(df_discount)
print(f"Loaded {len(df_discount)} discount records")


Loading discount data...
Loaded 11426 discount records


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [24]:
# =============================================================================
# Create pricing_with_discount DataFrame
# =============================================================================
print("Creating pricing_with_discount DataFrame...")

# Copy pricing_data
pricing_with_discount = pricing_data.copy()

# Merge discount data (by warehouse_id + product_id)
pricing_with_discount = pricing_with_discount.merge(
    df_discount[['warehouse_id', 'product_id', 'discount_perc']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing discount_perc with 0 (no discount)
pricing_with_discount['discount_perc'] = pricing_with_discount['discount_perc'].fillna(0)

# Merge all-time high margin data (P80 margin weighted by gross profit)
pricing_with_discount = pricing_with_discount.merge(
    df_all_time_high_margin[['warehouse_id', 'product_id', 'all_time_high_margin']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing all_time_high_margin with target_margin (fallback)
pricing_with_discount['all_time_high_margin'] = pricing_with_discount['all_time_high_margin'].fillna(
    pricing_with_discount['target_margin']
)

# =============================================================================
# Calculate price and margin after discount
# =============================================================================
# Price after discount = current_price * (1 - discount_perc)
pricing_with_discount['price_after_discount'] = (
    pricing_with_discount['current_price'] * (1 - pricing_with_discount['discount_perc'])
)

# Margin after discount = (price_after_discount - wac_p) / price_after_discount
pricing_with_discount['margin_after_discount'] = (
    (pricing_with_discount['price_after_discount'] - pricing_with_discount['wac_p']) / 
    pricing_with_discount['price_after_discount']
)

print(f"\n{'='*60}")
print(f"PRICING WITH DISCOUNT DATA COMPLETE")
print(f"{'='*60}")
print(f"Total records: {len(pricing_with_discount)}")
print(f"Records with discount (discount_perc > 0): {len(pricing_with_discount[pricing_with_discount['discount_perc'] > 0])}")
print(f"Records without discount: {len(pricing_with_discount[pricing_with_discount['discount_perc'] == 0])}")
print(f"\nNew columns added:")
print("  - discount_perc: discount percentage from sales")
print("  - price_after_discount: current_price * (1 - discount_perc)")
print("  - margin_after_discount: (price_after_discount - wac_p) / price_after_discount")
print(f"\nSample data with discounts:")
pricing_with_discount[pricing_with_discount['discount_perc'] > 0][
    ['product_id', 'warehouse_id', 'current_price', 'current_margin', 
     'discount_perc', 'price_after_discount', 'margin_after_discount']
].head(10)


Creating pricing_with_discount DataFrame...

PRICING WITH DISCOUNT DATA COMPLETE
Total records: 88626
Records with discount (discount_perc > 0): 2718
Records without discount: 85908

New columns added:
  - discount_perc: discount percentage from sales
  - price_after_discount: current_price * (1 - discount_perc)
  - margin_after_discount: (price_after_discount - wac_p) / price_after_discount

Sample data with discounts:


,product_id,warehouse_id,current_price,current_margin,discount_perc,price_after_discount,margin_after_discount
5,2424,337,178.25,0.048009,0.003046,177.707132,0.045101
21,958,339,281.25,0.026215,0.005337,279.749064,0.020991
35,1124,797,160.00,0.064905,0.003847,159.384516,0.061294
40,11490,339,295.00,0.083823,0.000477,294.859188,0.083385
49,8677,1,95.00,0.050422,0.002779,94.736000,0.047776
54,8635,1,105.00,0.041157,0.000344,104.963854,0.040827
56,205,337,286.75,0.060616,0.000458,286.618725,0.060186
57,205,8,286.75,0.060616,0.000183,286.697501,0.060444
69,20978,797,815.25,0.094013,0.018396,800.253063,0.077034
84,10574,170,60.25,0.082978,0.002384,60.106361,0.080787


In [25]:
# =============================================================================
# Price Position - Determine where price_after_discount falls in market tiers
# =============================================================================

def get_price_position(row):
    """Determine the price position relative to market price tiers."""
    price = row['price_after_discount']
    wac = row['wac_p']
    
    # Check if we have market data (minimum price exists)
    if pd.isna(row['minimum']) or pd.isna(price):
        return "No Market Data"
    
    # Get price tiers
    min_price = row['minimum']
    p25 = row['percentile_25']
    p50 = row['percentile_50']
    p75 = row['percentile_75']
    max_price = row['maximum']
    
    # Calculate below_market and above_market prices from margins
    # margin = (price - wac) / price  =>  price = wac / (1 - margin)
    below_market_margin = row['below_market']
    above_market_margin = row['above_market']
    
    below_market_price = wac / (1 - below_market_margin) if below_market_margin < 1 else min_price
    above_market_price = wac / (1 - above_market_margin) if above_market_margin < 1 else max_price
    
    # Determine position based on price tiers
    if price < below_market_price:
        return "Below Market"
    elif price < min_price:
        return "Below Min"
    elif price < p25:
        return "At Min"
    elif price < p50:
        return "At 25th"
    elif price < p75:
        return "At 50th"
    elif price < max_price:
        return "At 75th"
    elif price < above_market_price:
        return "At Max"
    else:
        return "Above Market"

# Apply price position function
pricing_with_discount['price_position'] = pricing_with_discount.apply(get_price_position, axis=1)

# Summary of price positions
print(f"\n{'='*60}")
print(f"PRICE POSITION ANALYSIS")
print(f"{'='*60}")
print("\nPrice Position Distribution:")
print(pricing_with_discount['price_position'].value_counts().to_string())
print(f"\nPrice Position Percentages:")
print((pricing_with_discount['price_position'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

# Sample data showing price position
print(f"\nSample data with price position:")
pricing_with_discount[
    ['product_id', 'warehouse_id', 'sku', 'current_price', 'discount_perc', 
     'price_after_discount', 'minimum', 'maximum', 'price_position']
].head(15)



PRICE POSITION ANALYSIS

Price Position Distribution:
price_position
No Market Data    31382
Below Market      17429
At Min            13923
Above Market      12626
At 25th            6162
At 50th            3844
At 75th            2929
At Max              331

Price Position Percentages:
price_position
No Market Data    35.41%
Below Market      19.67%
At Min            15.71%
Above Market      14.25%
At 25th            6.95%
At 50th            4.34%
At 75th             3.3%
At Max             0.37%
Name: proportion, dtype: object

Sample data with price position:


,product_id,warehouse_id,sku,current_price,discount_perc,price_after_discount,minimum,maximum,price_position
0,5811,339,بسكويت بيمبو بندق - 5 جنية,54.25,0.000000,54.250000,48.25,54.25,Above Market
1,5811,170,بسكويت بيمبو بندق - 5 جنية,54.25,0.000000,54.250000,48.25,54.25,Above Market
2,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,240.00,0.000000,240.000000,236.00,296.25,At Min
3,6224,501,بانيه كوكى توابل شرقية بارد 20 قطعة - 1 كجم,127.00,0.000000,127.000000,123.25,156.25,At Min
4,24709,401,بلـيزو سوداني محشو بالشوكولاتة مغطي بالسوداني ...,48.00,0.000000,48.000000,48.25,54.75,Below Market
5,2424,337,حفاضات جود كير مقاس 5 - 40 حفاضة,178.25,0.003046,177.707132,170.50,196.50,At Min
6,2424,8,حفاضات جود كير مقاس 5 - 40 حفاضة,178.25,0.000000,178.250000,170.50,196.50,At 25th
7,9153,703,زيت شعر فيانسيه مغذى باللوز - 175 مل,30.00,0.000000,30.000000,25.50,30.50,At 75th
8,2668,501,كلوركس جل انتعاش الليمون - 750 مل,497.00,0.000000,497.000000,604.00,676.50,Below Market
9,9288,501,مناديل بابيا نايلون 2 طبقة - 500 منديل,458.25,0.000000,458.250000,493.00,600.25,Below Market


In [26]:
# =============================================================================
# Stock Query - Get available stock by warehouse and product
# =============================================================================
STOCK_QUERY = '''
with parent_whs as (
select * 
from (
VALUES
(236,343),
(1,467),
(962,343)
)x(parent_id,child_id)

)
select warehouse_id,product_id,case when stocks_child is not null and stocks = 0 then stocks_child else stocks end as stocks 
from (
SELECT 
    pw.warehouse_id,
    pw.product_id,
    pw.available_stock::INTEGER AS stocks,
	pw2.available_stock::INTEGER AS stocks_child
	

FROM product_warehouse pw
left join parent_whs p on p.parent_id = pw.warehouse_id
left join product_warehouse pw2 on pw2.warehouse_id = p.child_id and pw.product_id = pw2.product_id
WHERE pw.warehouse_id NOT IN (6, 9, 10)
    AND pw.is_basic_unit = 1
)
'''

# Execute stock query
print("Loading stock data...")
df_stocks = query_snowflake(STOCK_QUERY)
df_stocks = convert_to_numeric(df_stocks)
print(f"Loaded {len(df_stocks)} stock records")

# Merge stock data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_stocks[['warehouse_id', 'product_id', 'stocks']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing stocks with 0
pricing_with_discount['stocks'] = pricing_with_discount['stocks'].fillna(0).astype(int)

print(f"\nStock data merged!")
print(f"Records with stock (stocks > 0): {len(pricing_with_discount[pricing_with_discount['stocks'] > 0])}")
print(f"Records without stock (stocks = 0): {len(pricing_with_discount[pricing_with_discount['stocks'] == 0])}")
print(f"\nSample data with stocks:")
pricing_with_discount[
    ['product_id', 'warehouse_id', 'sku', 'stocks', 'price_after_discount', 'price_position']
].head(10)


Loading stock data...


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 1968665 stock records

Stock data merged!
Records with stock (stocks > 0): 19451
Records without stock (stocks = 0): 80952

Sample data with stocks:


,product_id,warehouse_id,sku,stocks,price_after_discount,price_position
0,5811,339,بسكويت بيمبو بندق - 5 جنية,0,54.250000,Above Market
1,5811,170,بسكويت بيمبو بندق - 5 جنية,14,54.250000,Above Market
2,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,55,240.000000,At Min
3,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,55,240.000000,At Min
4,6224,501,بانيه كوكى توابل شرقية بارد 20 قطعة - 1 كجم,0,127.000000,At Min
5,24709,401,بلـيزو سوداني محشو بالشوكولاتة مغطي بالسوداني ...,0,48.000000,Below Market
6,2424,337,حفاضات جود كير مقاس 5 - 40 حفاضة,374,177.707132,At Min
7,2424,8,حفاضات جود كير مقاس 5 - 40 حفاضة,265,178.250000,At 25th
8,9153,703,زيت شعر فيانسيه مغذى باللوز - 175 مل,108,30.000000,At 75th
9,2668,501,كلوركس جل انتعاش الليمون - 750 مل,15,497.000000,Below Market


In [27]:
# =============================================================================
# Zero Demand Query - Identify SKUs with zero/low demand
# =============================================================================
ZERO_DEMAND_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
last_oss AS (
    SELECT product_id, warehouse_id, TIMESTAMP AS last_in_stock_day
    FROM (
        SELECT COALESCE(pwh.parent_id, sdc.warehouse_id) AS warehouse_id,
               sdc.product_id, sdc.TIMESTAMP,
               ROW_NUMBER() OVER(PARTITION BY COALESCE(pwh.parent_id, sdc.warehouse_id), sdc.product_id ORDER BY sdc.TIMESTAMP DESC) AS rnk 
        FROM materialized_views.STOCK_DAY_CLOSE sdc
        LEFT JOIN parent_whs pwh ON pwh.child_id = sdc.warehouse_id
        WHERE AVAILABLE_STOCK = 0 
            AND TIMESTAMP >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
        QUALIFY rnk = 1 
    )
),

current_stocks AS (
    SELECT COALESCE(pwh.parent_id, pw2.warehouse_id) AS warehouse_id,
           pw2.product_id, pw2.AVAILABLE_STOCK, pw2.activation
    FROM PRODUCT_WAREHOUSE pw2
    LEFT JOIN parent_whs pwh ON pwh.child_id = pw2.warehouse_id
    WHERE pw2.IS_BASIC_UNIT = 1
        AND CASE WHEN pw2.product_id = 1309 THEN pw2.packing_unit_id <> 23 ELSE TRUE END
),

prs AS (
    SELECT DISTINCT 
        product_purchased_receipts.product_id,
        COALESCE(pwh.parent_id, purchased_receipts.warehouse_id) AS warehouse_id,
        purchased_receipts.date::DATE AS date,
        product_purchased_receipts.purchased_item_count * product_purchased_receipts.basic_unit_count AS purchase_min_count
    FROM product_purchased_receipts
    JOIN purchased_receipts ON purchased_receipts.id = product_purchased_receipts.purchased_receipt_id
    LEFT JOIN parent_whs pwh ON pwh.child_id = purchased_receipts.warehouse_id
    JOIN last_oss lo ON product_purchased_receipts.product_id = lo.product_id 
        AND lo.warehouse_id = COALESCE(pwh.parent_id, purchased_receipts.warehouse_id) 
        AND purchased_receipts.date > lo.last_in_stock_day 
    WHERE product_purchased_receipts.purchased_item_count <> 0
        AND purchased_receipts.purchased_receipt_status_id IN (4, 5, 7)
        AND purchased_receipts.date::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
),

main AS (
    SELECT 
        prs.product_id, 
        prs.warehouse_id, 
        MIN(date) AS first_order_date, 
        SUM(purchase_min_count) AS total_recieved, 
        cs.AVAILABLE_STOCK, 
        cs.activation
    FROM prs 
    JOIN current_stocks cs ON cs.product_id = prs.product_id AND prs.warehouse_id = cs.warehouse_id
    GROUP BY prs.product_id, prs.warehouse_id, cs.AVAILABLE_STOCK, cs.activation
),

sold_days AS (
    SELECT product_id, warehouse_id, COUNT(DISTINCT o_date) AS sales_days
    FROM (
        SELECT DISTINCT
            so.created_at::DATE AS o_date,
            COALESCE(pwh.parent_id, pso.warehouse_id) AS warehouse_id,
            pso.product_id,
            SUM(pso.purchased_item_count * basic_unit_count) AS daily_qty
        FROM product_sales_order pso
        LEFT JOIN parent_whs pwh ON pwh.child_id = pso.warehouse_id
        JOIN sales_orders so ON so.id = pso.sales_order_id
        JOIN main m ON m.product_id = pso.product_id 
            AND m.warehouse_id = COALESCE(pwh.parent_id, pso.warehouse_id) 
            AND so.created_at::DATE >= m.first_order_date
        WHERE so.created_at::DATE BETWEEN CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120 
            AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
            AND pso.purchased_item_count <> 0
        GROUP BY o_date, COALESCE(pwh.parent_id, pso.warehouse_id), pso.product_id
    )
    GROUP BY product_id, warehouse_id
)

SELECT DISTINCT warehouse_id, product_id
FROM (
    SELECT m.product_id, m.warehouse_id, m.first_order_date, m.activation,
        COALESCE(sd.sales_days, 0) AS sales_days,
        COALESCE(sd.sales_days, 0)::FLOAT / NULLIF((CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1) - m.first_order_date, 0) AS perc_days
    FROM main m 
    LEFT JOIN sold_days sd ON sd.product_id = m.product_id AND sd.warehouse_id = m.warehouse_id
    WHERE m.first_order_date < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 10
)
WHERE perc_days <= 0.3
    AND activation = 'true'
'''

# Execute zero demand query
print("Loading zero demand SKUs...")
df_zero_demand = query_snowflake(ZERO_DEMAND_QUERY)
df_zero_demand = convert_to_numeric(df_zero_demand)
print(f"Loaded {len(df_zero_demand)} zero demand SKU records")


Loading zero demand SKUs...
Loaded 1346 zero demand SKU records


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [28]:
# =============================================================================
# Add Zero Demand Flag to pricing_with_discount
# =============================================================================

# Add a marker column to identify zero demand SKUs
df_zero_demand['zero_demand'] = 1

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_zero_demand[['warehouse_id', 'product_id', 'zero_demand']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values with 0 (not zero demand)
pricing_with_discount['zero_demand'] = pricing_with_discount['zero_demand'].fillna(0).astype(int)

print(f"Zero demand flag added!")
print(f"SKUs flagged as zero demand: {len(pricing_with_discount[pricing_with_discount['zero_demand'] == 1])}")
print(f"SKUs with normal demand: {len(pricing_with_discount[pricing_with_discount['zero_demand'] == 0])}")


Zero demand flag added!
SKUs flagged as zero demand: 1276
SKUs with normal demand: 99128


In [29]:
# =============================================================================
# OOS Yesterday Query - Identify SKUs out of stock yesterday
# =============================================================================
OOS_YESTERDAY_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT DISTINCT product_id, warehouse_id,
    CASE WHEN opening_stocks = 0 AND closing_stocks = 0 THEN 1
         ELSE 0 
    END AS oos_yesterday
FROM (
    SELECT 
        timestamp,
        product_id,
        COALESCE(pwh.parent_id, sdc.warehouse_id) AS warehouse_id, 
        AVAILABLE_STOCK AS closing_stocks,
        LAG(AVAILABLE_STOCK) OVER (PARTITION BY product_id, COALESCE(pwh.parent_id, sdc.warehouse_id) ORDER BY TIMESTAMP) AS opening_stocks
    FROM materialized_views.stock_day_close sdc
    LEFT JOIN parent_whs pwh ON pwh.child_id = sdc.warehouse_id
    WHERE timestamp::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 2
    QUALIFY opening_stocks IS NOT NULL
)
WHERE oos_yesterday = 1
'''

# Execute OOS yesterday query
print("Loading OOS yesterday data...")
df_oos_yesterday = query_snowflake(OOS_YESTERDAY_QUERY)
df_oos_yesterday = convert_to_numeric(df_oos_yesterday)
print(f"Loaded {len(df_oos_yesterday)} OOS yesterday records")


Loading OOS yesterday data...
Loaded 1942001 OOS yesterday records


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [30]:
# =============================================================================
# Add OOS Yesterday Flag to pricing_with_discount
# =============================================================================

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_oos_yesterday[['warehouse_id', 'product_id', 'oos_yesterday']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values with 0 (not OOS yesterday)
pricing_with_discount['oos_yesterday'] = pricing_with_discount['oos_yesterday'].fillna(0).astype(int)

print(f"OOS yesterday flag added!")
print(f"SKUs out of stock yesterday: {len(pricing_with_discount[pricing_with_discount['oos_yesterday'] == 1])}")
print(f"SKUs in stock yesterday: {len(pricing_with_discount[pricing_with_discount['oos_yesterday'] == 0])}")


OOS yesterday flag added!
SKUs out of stock yesterday: 80036
SKUs in stock yesterday: 20368


In [31]:
# =============================================================================
# Running Rate Query - Recent-demand-focused (30-day primary, 90-day fallback)
# Exponential decay weighting with parent warehouse mapping
# =============================================================================
RUNNING_RATE_QUERY = f'''
WITH params AS (
  SELECT
    CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS run_date,
    30 AS main_window_days,
    90 AS fallback_window_days,
    7.0 AS half_life_days,
    14.0 AS fallback_half_life_days,
    3 AS min_days_threshold
),

-- Parent/child warehouse mapping (chilled warehouses -> main)
parent_whs AS (
  SELECT * FROM (
    VALUES (236, 343), (1, 467), (962, 343)
  ) x(parent_id, child_id)
),

-- 1) Raw daily sales (last 90 days)
raw_daily_sales AS (
  SELECT
    pso.product_id,
    pso.warehouse_id,
    CAST(DATE_TRUNC('day', pso.created_at) AS DATE) AS sale_date,
    SUM(pso.purchased_item_count * pso.basic_unit_count) AS sold_units
  FROM product_sales_order pso
  JOIN sales_orders so ON pso.sales_order_id = so.id
  WHERE CAST(DATE_TRUNC('day', pso.created_at) AS DATE)
        >= DATEADD(day, -90, (SELECT run_date FROM params))
    AND CAST(DATE_TRUNC('day', pso.created_at) AS DATE) < (SELECT run_date FROM params)
    AND so.sales_order_status_id NOT IN (7, 12)
    AND so.channel IN ('telesales', 'retailer')
    AND pso.purchased_item_count <> 0
  GROUP BY 1, 2, 3
),

-- 2) Merge child warehouse sales into parent
daily_sales AS (
  SELECT
    product_id,
    warehouse_id,
    sale_date,
    SUM(sold_units) AS sold_units
  FROM (
    SELECT product_id, warehouse_id, sale_date, sold_units
    FROM raw_daily_sales

    UNION ALL

    SELECT r.product_id, p.parent_id AS warehouse_id, r.sale_date, r.sold_units
    FROM raw_daily_sales r
    JOIN parent_whs p ON r.warehouse_id = p.child_id
  )
  GROUP BY 1, 2, 3
),

-- 3) In-stock days from stock_day_close (last 90 days)
in_stock_days AS (
  SELECT
    base.product_id,
    base.warehouse_id,
    base.stock_date,
    DATEDIFF('day', base.stock_date, (SELECT run_date FROM params)) AS days_ago
  FROM (
    SELECT
      dc.product_id,
      dc.warehouse_id,
      CAST(DATE_TRUNC('day', dc.timestamp) AS DATE) AS stock_date,
      MAX(dc.available_stock) AS parent_stock
    FROM materialized_views.STOCK_DAY_CLOSE dc
    WHERE CAST(DATE_TRUNC('day', dc.timestamp) AS DATE)
          >= DATEADD(day, -90,
               CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
      AND CAST(DATE_TRUNC('day', dc.timestamp) AS DATE)
          < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
      AND dc.product_id IS NOT NULL
    GROUP BY 1, 2, 3
  ) base
  LEFT JOIN parent_whs p ON base.warehouse_id = p.parent_id
  LEFT JOIN (
    SELECT product_id, CAST(DATE_TRUNC('day', timestamp) AS DATE) AS stock_date,
           warehouse_id, MAX(available_stock) AS child_stock
    FROM materialized_views.STOCK_DAY_CLOSE
    WHERE CAST(DATE_TRUNC('day', timestamp) AS DATE)
          >= DATEADD(day, -90,
               CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
      AND CAST(DATE_TRUNC('day', timestamp) AS DATE)
          < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
    GROUP BY 1, 2, 3
  ) child_stock
    ON child_stock.warehouse_id = p.child_id
   AND child_stock.product_id = base.product_id
   AND child_stock.stock_date = base.stock_date
  WHERE GREATEST(COALESCE(base.parent_stock, 0), COALESCE(child_stock.child_stock, 0)) > 0
),

-- 4) In-stock sales: join stock days with sales (0 if no sale that day)
in_stock_sales AS (
  SELECT
    isd.product_id,
    isd.warehouse_id,
    isd.stock_date,
    isd.days_ago,
    COALESCE(ds.sold_units, 0) AS sold_units,
    CASE WHEN isd.days_ago <= 30 THEN 1 ELSE 0 END AS in_recent_window
  FROM in_stock_days isd
  LEFT JOIN daily_sales ds
    ON isd.product_id = ds.product_id
   AND isd.warehouse_id = ds.warehouse_id
   AND isd.stock_date = ds.sale_date
),

-- SKUs in stock all last 3 days with zero sales → dead demand
recent_zero_sales AS (
  SELECT
    product_id,
    warehouse_id
  FROM in_stock_sales
  WHERE days_ago BETWEEN 1 AND 4
  GROUP BY 1, 2
  HAVING COUNT(*) = 4 AND SUM(sold_units) = 0
),

-- 5) Count recent in-stock days per SKU (to decide primary vs fallback)
sku_recent_days AS (
  SELECT
    product_id,
    warehouse_id,
    SUM(in_recent_window) AS recent_in_stock_days
  FROM in_stock_sales
  GROUP BY 1, 2
),

-- 6) P95 cap per SKU (across all available in-stock days)
outlier_caps AS (
  SELECT
    product_id,
    warehouse_id,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY sold_units) AS p95
  FROM in_stock_sales
  GROUP BY 1, 2
),

-- 7) Apply capping + exponential decay weight
cleaned_weighted AS (
  SELECT
    s.product_id,
    s.warehouse_id,
    LEAST(s.sold_units, GREATEST(oc.p95, 1)) AS capped_units,
    s.days_ago,
    rd.recent_in_stock_days,
    CASE
      WHEN rd.recent_in_stock_days >= (SELECT min_days_threshold FROM params)
        THEN EXP(-0.693 * s.days_ago / (SELECT half_life_days FROM params))
      ELSE EXP(-0.693 * s.days_ago / (SELECT fallback_half_life_days FROM params))
    END AS decay_weight
  FROM in_stock_sales s
  JOIN outlier_caps oc
    ON s.product_id = oc.product_id AND s.warehouse_id = oc.warehouse_id
  JOIN sku_recent_days rd
    ON s.product_id = rd.product_id AND s.warehouse_id = rd.warehouse_id
  WHERE
    (rd.recent_in_stock_days >= (SELECT min_days_threshold FROM params)
       AND s.in_recent_window = 1)
    OR
    (rd.recent_in_stock_days < (SELECT min_days_threshold FROM params))
),

-- 8) Weighted running rate
forecast AS (
  SELECT
    product_id,
    warehouse_id,
    CEIL(SUM(capped_units * decay_weight) / NULLIF(SUM(decay_weight), 0)) AS in_stock_rr,
    COUNT(*) AS n_in_stock_days,
    MAX(recent_in_stock_days) AS recent_in_stock_days
  FROM cleaned_weighted
  GROUP BY 1, 2
)

SELECT
  f.product_id,
  f.warehouse_id,
  CASE
    WHEN f.n_in_stock_days < 3 THEN 0
    WHEN rz.product_id IS NOT NULL THEN 0
    ELSE f.in_stock_rr
  END AS in_stock_rr
FROM forecast f
LEFT JOIN recent_zero_sales rz
  ON rz.product_id = f.product_id
 AND rz.warehouse_id = f.warehouse_id
'''

# Execute running rate query
print("Loading running rate data (this may take a moment)...")
df_running_rate = query_snowflake(RUNNING_RATE_QUERY)
df_running_rate = convert_to_numeric(df_running_rate)
print(f"Loaded {len(df_running_rate)} running rate records")


Loading running rate data (this may take a moment)...
Loaded 31664 running rate records


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [32]:
# =============================================================================
# Merge Running Rate and Calculate DOH (Days on Hand)
# =============================================================================

# Merge running rate data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_running_rate[['warehouse_id', 'product_id', 'in_stock_rr']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values for running rate columns
pricing_with_discount['in_stock_rr'] = pricing_with_discount['in_stock_rr'].fillna(0)

# Calculate DOH (Days on Hand) = stocks / in_stock_rr
# Handle division by zero - if running rate is 0, DOH is infinite (use 999)
pricing_with_discount['doh'] = np.select(
    [
        (pricing_with_discount['in_stock_rr'] > 0) & (pricing_with_discount['stocks'] > 0),
        pricing_with_discount['stocks'] == 0
    ],
    [
        pricing_with_discount['stocks'] / pricing_with_discount['in_stock_rr'],
        0
    ],
    default=999
)


In [33]:
# =============================================================================
# Product Classification Query - ABC Classification based on order contribution
# =============================================================================
PRODUCT_CLASSIFICATION_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
order_counts AS (
    SELECT 
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        COUNT(DISTINCT pso.sales_order_id) AS order_count
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 90
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), pso.product_id
),

warehouse_totals AS (
    SELECT 
        warehouse_id,
        SUM(order_count) AS total_orders
    FROM order_counts
    GROUP BY warehouse_id
),

ranked_products AS (
    SELECT 
        oc.warehouse_id,
        oc.product_id,
        oc.order_count,
        wt.total_orders,
        oc.order_count::FLOAT / NULLIF(wt.total_orders, 0) AS contribution,
        SUM(oc.order_count::FLOAT / NULLIF(wt.total_orders, 0)) 
            OVER (PARTITION BY oc.warehouse_id ORDER BY oc.order_count DESC 
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_contribution
    FROM order_counts oc
    JOIN warehouse_totals wt ON oc.warehouse_id = wt.warehouse_id
)

SELECT 
    warehouse_id,
    product_id,
    order_count,
    contribution,
    cumulative_contribution,
    CASE 
        WHEN cumulative_contribution <= 0.3 THEN 'A'
        WHEN cumulative_contribution <= 0.75 THEN 'B'
        ELSE 'C'
    END AS abc_class
FROM ranked_products
'''

# Execute product classification query
print("Loading product classification data...")
df_classification = query_snowflake(PRODUCT_CLASSIFICATION_QUERY)
df_classification = convert_to_numeric(df_classification)
print(f"Loaded {len(df_classification)} product classification records")
print(f"\nClassification distribution:")
print(df_classification['abc_class'].value_counts().to_string())


Loading product classification data...
Loaded 29228 product classification records

Classification distribution:
abc_class
C    23516
B     4942
A      770


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [34]:
# =============================================================================
# Add ABC Classification to pricing_with_discount
# =============================================================================

# Merge classification data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_classification[['warehouse_id', 'product_id', 'order_count', 'contribution', 'abc_class']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values - products without orders in last 3 months get class 'C'
pricing_with_discount['order_count'] = pricing_with_discount['order_count'].fillna(0).astype(int)
pricing_with_discount['contribution'] = pricing_with_discount['contribution'].fillna(0)
pricing_with_discount['abc_class'] = pricing_with_discount['abc_class'].fillna('C')

print(f"ABC Classification added!")
print(f"\nClassification in pricing_with_discount:")
print(pricing_with_discount['abc_class'].value_counts().to_string())
print(f"\nSample data with classification:")
pricing_with_discount[
    ['product_id', 'warehouse_id', 'sku', 'order_count', 'contribution', 'abc_class', 'stocks', 'doh']
].head(15)


ABC Classification added!

Classification in pricing_with_discount:
abc_class
C    93904
B     5655
A      845

Sample data with classification:


,product_id,warehouse_id,sku,order_count,contribution,abc_class,stocks,doh
0,5811,339,بسكويت بيمبو بندق - 5 جنية,235,0.000829,B,0,0.000000
1,5811,170,بسكويت بيمبو بندق - 5 جنية,179,0.000740,B,14,1.166667
2,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,203,0.000228,C,55,13.750000
3,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,203,0.000228,C,55,13.750000
4,6224,501,بانيه كوكى توابل شرقية بارد 20 قطعة - 1 كجم,0,0.000000,C,0,0.000000
5,24709,401,بلـيزو سوداني محشو بالشوكولاتة مغطي بالسوداني ...,0,0.000000,C,0,0.000000
6,2424,337,حفاضات جود كير مقاس 5 - 40 حفاضة,624,0.002604,B,374,5.843750
7,2424,8,حفاضات جود كير مقاس 5 - 40 حفاضة,834,0.002415,B,265,1.568047
8,9153,703,زيت شعر فيانسيه مغذى باللوز - 175 مل,32,0.000155,C,108,27.000000
9,2668,501,كلوركس جل انتعاش الليمون - 750 مل,6,0.000038,C,15,999.000000


In [35]:
# =============================================================================
# PO (Purchase Order) Data Query - Last PO status and rejection count
# =============================================================================
PO_DATA_QUERY = '''
WITH last_data AS (
    SELECT product_id, warehouse_id, confirmation_status, PO_date::DATE AS last_po_date, ordered_qty
    FROM (
        SELECT 
            product_id,
            Target_WAREHOUSE_ID AS warehouse_id,
            confirmation_status,
            created_at AS PO_date,
            MIN_QUANTITY AS ordered_qty,
            reason,
            MAX(created_at) OVER (PARTITION BY product_id, Target_WAREHOUSE_ID) AS last_po
        FROM retool.PO_INITIAL_PLAN
        WHERE created_at::DATE >= CURRENT_DATE - 15 
    ) x
    WHERE last_po = PO_date
),

last_15_data AS (
    SELECT 
        product_id,
        target_WAREHOUSE_ID AS warehouse_id,
        COUNT(DISTINCT CASE WHEN confirmation_status <> 'yes' THEN created_at END) AS no_last_15
    FROM retool.PO_INITIAL_PLAN
    WHERE created_at::DATE >= CURRENT_DATE - 15 
    GROUP BY 1, 2
)

SELECT 
    ld.product_id,
    ld.warehouse_id,
    ld.confirmation_status,
    ld.last_po_date,
    ld.ordered_qty,
    COALESCE(lfd.no_last_15, 0) AS no_last_15
FROM last_data ld 
LEFT JOIN last_15_data lfd 
    ON lfd.product_id = ld.product_id 
    AND lfd.warehouse_id = ld.warehouse_id
'''

# Execute PO data query using dwh_pg_query
print("Loading PO data...")
df_po_data = setup_environment_2.dwh_pg_query(
    PO_DATA_QUERY, 
    columns=['product_id', 'warehouse_id', 'confirmation_status', 'last_po_date', 'ordered_qty', 'no_last_15']
)
df_po_data.columns = df_po_data.columns.str.lower()
df_po_data = convert_to_numeric(df_po_data)
print(f"Loaded {len(df_po_data)} PO records")
print(f"\nConfirmation status distribution:")
print(df_po_data['confirmation_status'].value_counts().to_string())


Loading PO data...
Loaded 18887 PO records

Confirmation status distribution:
confirmation_status
yes    13634
no      4971


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [36]:
# =============================================================================
# Add PO Data to pricing_with_discount
# =============================================================================

# Merge PO data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_po_data[['warehouse_id', 'product_id', 'confirmation_status', 'last_po_date', 'ordered_qty', 'no_last_15']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values
pricing_with_discount['ordered_qty'] = pricing_with_discount['ordered_qty'].fillna(0)
pricing_with_discount['no_last_15'] = pricing_with_discount['no_last_15'].fillna(0).astype(int)

print(f"PO data added!")
print(f"\nRecords with PO data: {len(pricing_with_discount[~pricing_with_discount['confirmation_status'].isna()])}")
print(f"Records without PO data: {len(pricing_with_discount[pricing_with_discount['confirmation_status'].isna()])}")
print(f"\nSample data with PO info:")
pricing_with_discount[
    ['product_id', 'warehouse_id', 'sku', 'confirmation_status', 'last_po_date', 'ordered_qty', 'no_last_15']
].dropna(subset=['confirmation_status']).head(15)


PO data added!

Records with PO data: 21194
Records without PO data: 79210

Sample data with PO info:


,product_id,warehouse_id,sku,confirmation_status,last_po_date,ordered_qty,no_last_15
0,5811,339,بسكويت بيمبو بندق - 5 جنية,no,2026-04-19,324.0,3
1,5811,170,بسكويت بيمبو بندق - 5 جنية,no,2026-04-19,120.0,1
2,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,yes,2026-04-15,24.0,0
3,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,yes,2026-04-15,24.0,0
6,2424,337,حفاضات جود كير مقاس 5 - 40 حفاضة,no,2026-04-19,185.0,6
7,2424,8,حفاضات جود كير مقاس 5 - 40 حفاضة,no,2026-04-19,2135.0,6
8,9153,703,زيت شعر فيانسيه مغذى باللوز - 175 مل,yes,2026-04-16,108.0,0
14,1700,1,مناديل فاميليا سحب 2 طبقة - 500 منديل,yes,2026-04-18,12.0,10
17,25718,632,صلصة رشيدي الميزان برطمان - 300 جم,yes,2026-04-07,7.0,0
19,9609,337,سمن جنة - 2.5 كجم,yes,2026-04-11,14.0,0


In [37]:
# =============================================================================
# Leadtime Query - Supplier leadtime by brand, category, and warehouse
# =============================================================================
LEADTIME_QUERY = '''
SELECT brand, cat, warehouse_id, leadtime
FROM (
    SELECT a.*, b.name_ar AS brand, c.name_ar AS cat
    FROM (
        SELECT DISTINCT 
            sl.supplier_id, 
            warehouse_id, 
            category_id, 
            brand_id, 
            sl.updated_at, 
            leadtime,
            MAX(sl.updated_at) OVER (PARTITION BY sl.supplier_id, warehouse_id) AS last_update
        FROM retool.SUPPLIER_MOQ sl 
        JOIN retool.PO_SUPPLIER_MAPPING sm ON sl.supplier_id = sm.supplier_id 
    ) a
    JOIN brands b ON b.id = a.brand_id 
    JOIN categories c ON c.id = a.category_id
    WHERE a.updated_at = last_update
) d
'''

# Execute leadtime query using dwh_pg_query
print("Loading leadtime data...")
df_leadtime = setup_environment_2.dwh_pg_query(
    LEADTIME_QUERY, 
    columns=['brand', 'cat', 'warehouse_id', 'leadtime']
)
df_leadtime.columns = df_leadtime.columns.str.lower()
df_leadtime = convert_to_numeric(df_leadtime)
print(f"Loaded {len(df_leadtime)} leadtime records")


Loading leadtime data...
Loaded 15294 leadtime records


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [38]:
# =============================================================================
# Add Leadtime to pricing_with_discount
# =============================================================================

# Merge leadtime data with pricing_with_discount (by brand, cat, warehouse_id)
pricing_with_discount = pricing_with_discount.merge(
    df_leadtime[['brand', 'cat', 'warehouse_id', 'leadtime']], 
    on=['brand', 'cat', 'warehouse_id'], 
    how='left'
)

# Fill missing leadtime with 0 or a default value
pricing_with_discount['leadtime'] = pricing_with_discount['leadtime'].fillna(72)


print(f"Leadtime data added!")
print(f"\nRecords with leadtime: {len(pricing_with_discount[pricing_with_discount['leadtime'] > 0])}")
print(f"Records without leadtime: {len(pricing_with_discount[pricing_with_discount['leadtime'] == 0])}")
print(f"\nLeadtime distribution:")
print(pricing_with_discount['leadtime'].describe())

# =============================================================================
# Calculate Expected Receiving Day
# If confirmation_status is 'no': add 2 extra days (48 hours) before adding leadtime
# expected_receiving_day = last_po_date + ((2 + leadtime) / 24) if not confirmed
# expected_receiving_day = last_po_date + (leadtime / 24) if confirmed
# =============================================================================

# Convert last_po_date to datetime if not already
pricing_with_discount['last_po_date'] = pd.to_datetime(pricing_with_discount['last_po_date'], errors='coerce')

# Calculate adjusted leadtime: add 48 hours (2 days) if confirmation_status is 'no'
pricing_with_discount['adjusted_leadtime'] = np.where(
    pricing_with_discount['confirmation_status'].str.lower() == 'no',
    pricing_with_discount['leadtime'] + 48,  # Add 2 days (48 hours) if not confirmed
    pricing_with_discount['leadtime']
)

# Calculate expected receiving day (leadtime is in hours, divide by 24 for days)
pricing_with_discount['expected_receiving_day'] = pricing_with_discount['last_po_date'] + pd.to_timedelta(
    pricing_with_discount['adjusted_leadtime'] / 24, unit='D'
)

# Set expected_receiving_day to empty if it's in the past (smaller than today)
pricing_with_discount['expected_receiving_day'] = np.where(
    pricing_with_discount['expected_receiving_day'] < pd.Timestamp(TODAY),
    pd.NaT,
    pricing_with_discount['expected_receiving_day']
)
# Convert back to datetime (np.where returns object type)
pricing_with_discount['expected_receiving_day'] = pd.to_datetime(pricing_with_discount['expected_receiving_day'])

print(f"\nExpected receiving day calculated!")
print(f"Records with expected receiving day (future dates only): {len(pricing_with_discount[~pricing_with_discount['expected_receiving_day'].isna()])}")
print(f"Records with past expected dates (set to empty): {len(pricing_with_discount[pricing_with_discount['expected_receiving_day'].isna() & pricing_with_discount['last_po_date'].notna()])}")
print(f"Records with confirmation_status='no' (added 2 extra days): {len(pricing_with_discount[pricing_with_discount['confirmation_status'].str.lower() == 'no'])}")
print(f"\nSample data with expected receiving day:")
pricing_with_discount[~pricing_with_discount['last_po_date'].isna()][
    ['product_id', 'warehouse_id', 'sku', 'confirmation_status', 'last_po_date', 'leadtime', 'adjusted_leadtime', 'expected_receiving_day', 'doh']
].head(15)


Leadtime data added!

Records with leadtime: 108435
Records without leadtime: 0

Leadtime distribution:
count    108435.000000
mean         54.470826
std          30.046827
min          24.000000
25%          48.000000
50%          48.000000
75%          72.000000
max         168.000000
Name: leadtime, dtype: float64

Expected receiving day calculated!
Records with expected receiving day (future dates only): 13187
Records with past expected dates (set to empty): 10064
Records with confirmation_status='no' (added 2 extra days): 5734

Sample data with expected receiving day:


,product_id,warehouse_id,sku,confirmation_status,last_po_date,leadtime,adjusted_leadtime,expected_receiving_day,doh
0,5811,339,بسكويت بيمبو بندق - 5 جنية,no,2026-04-19,48.0,96.0,2026-04-23,0.000000
1,5811,170,بسكويت بيمبو بندق - 5 جنية,no,2026-04-19,48.0,96.0,2026-04-23,1.166667
2,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,yes,2026-04-15,48.0,48.0,NaT,13.750000
3,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,yes,2026-04-15,48.0,48.0,NaT,13.750000
6,2424,337,حفاضات جود كير مقاس 5 - 40 حفاضة,no,2026-04-19,24.0,72.0,2026-04-22,5.843750
7,2424,8,حفاضات جود كير مقاس 5 - 40 حفاضة,no,2026-04-19,24.0,72.0,2026-04-22,1.568047
8,9153,703,زيت شعر فيانسيه مغذى باللوز - 175 مل,yes,2026-04-16,48.0,48.0,NaT,27.000000
11,4076,339,شوكولاتة جلاكسى سادة - 18 جم,None,2026-04-18,48.0,48.0,2026-04-20,0.000000
12,4076,170,شوكولاتة جلاكسى سادة - 18 جم,None,2026-04-18,48.0,48.0,2026-04-20,0.000000
14,1700,1,مناديل فاميليا سحب 2 طبقة - 500 منديل,yes,2026-04-18,24.0,24.0,2026-04-19,0.000000


In [39]:
# =============================================================================
# SKIP: Margin Boundaries Query - Now fetched via market_data_module.get_margin_tiers()
# =============================================================================
# The margin boundaries and tier calculation is now centralized in market_data_module.
# We'll use get_margin_tiers() to get pre-calculated margin tiers.

print("Loading margin tiers from market_data_module...")
df_margin_tiers = get_margin_tiers()
print(f"Loaded {len(df_margin_tiers)} margin tier records from module")


Loading margin tiers from market_data_module...

FETCHING MARGIN TIERS
Timestamp: 2026-04-19 14:16:33 Cairo time

Step 1: Computing margin boundaries from sales data...
    Loaded 37427 records (per product per warehouse)

Step 2: Mapping warehouses to cohorts...
    Records after dedup: 37427

Step 2a: Building scaffold of all active product-warehouse pairs...
    Scaffold: 43898 active pairs, added 6471 rows without warehouse-level boundaries

Step 2b: Cascading fallback for bad boundaries...
    8334 product-warehouse rows need fallback (both < 0 or missing)
Fetching region-level margin boundaries...
  Loaded 19148 product-region margin boundary records
    After region fallback: 6307 still bad
Fetching global-level margin boundaries...
  Loaded 4297 product-level margin boundary records
    After global fallback: 2980 still bad
    Fallback summary: 2027 region, 6307 global

Step 3: Calculating margin tiers...

MARGIN TIERS COMPLETE
Total records: 43898
Loaded 43898 margin tier rec

In [40]:
# =============================================================================
# Add Margin Tiers from market_data_module (Pre-calculated)
# =============================================================================
# The margin tiers are now calculated in market_data_module.get_margin_tiers()
# We just need to merge them with pricing_with_discount

# Merge pre-calculated margin tiers (by warehouse_id + product_id)
pricing_with_discount = pricing_with_discount.merge(
    df_margin_tiers[[
        'product_id', 'warehouse_id',
        'optimal_bm', 'min_boundary', 'max_boundary', 'median_bm',
        'effective_min_margin', 'margin_step',
        'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
        'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2'
    ]], 
    on=['warehouse_id', 'product_id'],
    how='left'
)

print(f"Margin tiers merged from module!")
print(f"\nRecords with margin tiers: {len(pricing_with_discount[~pricing_with_discount['max_boundary'].isna()])}")
print(f"Records without margin tiers: {len(pricing_with_discount[pricing_with_discount['max_boundary'].isna()])}")

print(f"\nMargin Tier Structure (from market_data_module):")
print(f"  margin_tier_below:   effective_min - step (1 below)")
print(f"  margin_tier_1:       effective_min_margin")
print(f"  margin_tier_2:       effective_min + 1*step")
print(f"  margin_tier_3:       effective_min + 2*step")
print(f"  margin_tier_4:       effective_min + 3*step")
print(f"  margin_tier_5:       max_boundary")
print(f"  margin_tier_above_1: max_boundary + 1*step")
print(f"  margin_tier_above_2: max_boundary + 2*step")

print(f"\nSample margin tiers:")
pricing_with_discount[~pricing_with_discount['max_boundary'].isna()][
    ['product_id', 'sku', 'effective_min_margin', 'max_boundary', 'margin_step',
     'margin_tier_below', 'margin_tier_1', 'margin_tier_3', 'margin_tier_5', 
     'margin_tier_above_1', 'margin_tier_above_2']
].head(10)


Margin tiers merged from module!

Records with margin tiers: 45494
Records without margin tiers: 62941

Margin Tier Structure (from market_data_module):
  margin_tier_below:   effective_min - step (1 below)
  margin_tier_1:       effective_min_margin
  margin_tier_2:       effective_min + 1*step
  margin_tier_3:       effective_min + 2*step
  margin_tier_4:       effective_min + 3*step
  margin_tier_5:       max_boundary
  margin_tier_above_1: max_boundary + 1*step
  margin_tier_above_2: max_boundary + 2*step

Sample margin tiers:


,product_id,sku,effective_min_margin,max_boundary,margin_step,margin_tier_below,margin_tier_1,margin_tier_3,margin_tier_5,margin_tier_above_1,margin_tier_above_2
0,5811,بسكويت بيمبو بندق - 5 جنية,0.005321,0.078662,0.018335,0.005321,0.014489,0.041992,0.078662,0.096997,0.115332
1,5811,بسكويت بيمبو بندق - 5 جنية,0.003101,0.075345,0.018061,0.003101,0.012132,0.039223,0.075345,0.093406,0.111467
2,10594,كورونا شوكولاتة داكنة 72%- 45 جنية,0.018760,0.064391,0.011408,0.018760,0.024464,0.041576,0.064391,0.075799,0.087207
3,10594,كورونا شوكولاتة داكنة 72%- 45 جنية,0.018760,0.064391,0.011408,0.018760,0.024464,0.041576,0.064391,0.075799,0.087207
6,2424,حفاضات جود كير مقاس 5 - 40 حفاضة,0.009862,0.078435,0.017143,0.009862,0.018434,0.044149,0.078435,0.095578,0.112721
7,2424,حفاضات جود كير مقاس 5 - 40 حفاضة,0.016425,0.076598,0.015043,0.016425,0.023946,0.046511,0.076598,0.091641,0.106685
8,9153,زيت شعر فيانسيه مغذى باللوز - 175 مل,0.021341,0.066713,0.011343,0.021341,0.027012,0.044027,0.066713,0.078056,0.089398
9,2668,كلوركس جل انتعاش الليمون - 750 مل,0.043925,0.103578,0.014913,0.043925,0.051382,0.073752,0.103578,0.118491,0.133404
11,4076,شوكولاتة جلاكسى سادة - 18 جم,0.086396,0.126456,0.010015,0.086396,0.091404,0.106426,0.126456,0.136471,0.146486
12,4076,شوكولاتة جلاكسى سادة - 18 جم,0.081470,0.120450,0.009745,0.081470,0.086343,0.100960,0.120450,0.130195,0.139940


In [41]:
# =============================================================================
# Minimum Selling Quantity Query - Get min selling qty per product
# =============================================================================
MIN_SELLING_QTY_QUERY = f'''
SELECT product_id, min_selling_qty
FROM (
    SELECT *, MIN(basic_unit_count) OVER (PARTITION BY product_id) AS min_selling_qty
    FROM (
        SELECT DISTINCT
            pso.product_id,
            pso.PACKING_UNIT_ID,
            pup.basic_unit_count,
            SUM(pso.total_price) AS nmv,
            SUM(pso.total_price) / SUM(nmv) OVER (PARTITION BY pso.product_id) AS cntrb
        FROM product_sales_order pso
        JOIN PACKING_UNIT_PRODUCTS pup ON pup.product_id = pso.product_id 
            AND pup.PACKING_UNIT_ID = pso.PACKING_UNIT_ID
        JOIN sales_orders so ON so.id = pso.sales_order_id
        WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
            AND pso.purchased_item_count <> 0
        GROUP BY ALL
        QUALIFY cntrb > 0.05
    )
    QUALIFY basic_unit_count = min_selling_qty
)
'''

# Execute min selling qty query
print("Loading minimum selling quantity data...")
df_min_selling_qty = query_snowflake(MIN_SELLING_QTY_QUERY)
df_min_selling_qty = convert_to_numeric(df_min_selling_qty)
print(f"Loaded {len(df_min_selling_qty)} min selling qty records")


Loading minimum selling quantity data...
Loaded 3964 min selling qty records


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [42]:
# =============================================================================
# Add Min Selling Qty and Below Min Stock Flag to pricing_with_discount
# =============================================================================

# Merge min selling qty with pricing_with_discount (by product_id)
pricing_with_discount = pricing_with_discount.merge(
    df_min_selling_qty[['product_id', 'min_selling_qty']], 
    on='product_id', 
    how='left'
)

# Fill missing min_selling_qty with 1 (default)
pricing_with_discount['min_selling_qty'] = pricing_with_discount['min_selling_qty'].fillna(1).astype(int)

# Create flag: below_min_stock_flag = 1 if (RR = 0 AND stocks > 0 AND stocks < min_selling_qty)
pricing_with_discount['below_min_stock_flag'] = np.where(
    (pricing_with_discount['in_stock_rr'] == 0) & 
    (pricing_with_discount['stocks'] > 0) &
    (pricing_with_discount['stocks'] < pricing_with_discount['min_selling_qty']),
    1, 0
)

print(f"Min selling qty and below_min_stock_flag added!")
print(f"\nSKUs flagged (zero RR & stocks < min_selling_qty): {len(pricing_with_discount[pricing_with_discount['below_min_stock_flag'] == 1])}")
print(f"SKUs not flagged: {len(pricing_with_discount[pricing_with_discount['below_min_stock_flag'] == 0])}")
print(f"\nSample flagged SKUs:")
pricing_with_discount[pricing_with_discount['below_min_stock_flag'] == 1][
    ['product_id', 'warehouse_id', 'sku', 'stocks', 'min_selling_qty', 'in_stock_rr', 'below_min_stock_flag']
].head(15)


Min selling qty and below_min_stock_flag added!

SKUs flagged (zero RR & stocks < min_selling_qty): 70
SKUs not flagged: 108365

Sample flagged SKUs:


,product_id,warehouse_id,sku,stocks,min_selling_qty,in_stock_rr,below_min_stock_flag
1134,3558,962,تونة صن شاين اكسبريس مفتتة بارد - 150 جم,3,12,0.0,1
1135,3558,962,تونة صن شاين اكسبريس مفتتة بارد - 150 جم,3,12,0.0,1
1136,3558,962,تونة صن شاين اكسبريس مفتتة بارد - 150 جم,3,12,0.0,1
1137,3558,962,تونة صن شاين اكسبريس مفتتة بارد - 150 جم,3,12,0.0,1
3526,2859,703,تونة دولفين مفتتة حار 170 جم + 30 جم - 200 جم,2,12,0.0,1
5264,12470,236,مولبد طويل جدا روز مسك ماكسي مضغوط 34 + 6 فوط...,2,3,0.0,1
5265,12470,236,مولبد طويل جدا روز مسك ماكسي مضغوط 34 + 6 فوط...,2,3,0.0,1
5266,12470,236,مولبد طويل جدا روز مسك ماكسي مضغوط 34 + 6 فوط...,2,3,0.0,1
5934,4674,797,التمساح عسل اسود - 680 جم,2,3,0.0,1
5935,4674,797,التمساح عسل اسود - 680 جم,2,3,0.0,1


In [43]:
# =============================================================================
# Yesterday's Discount Analysis Query
# Gets: SKU discount, Quantity discount, Tier 1/2/3 NMV breakdown and contributions
# =============================================================================
YESTERDAY_DISCOUNT_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
qd_det AS (
    -- Map dynamic tags to warehouse IDs using name matching
    SELECT DISTINCT 
        dt.id AS tag_id, 
        dt.name AS tag_name,
        REPLACE(w.name, ' ', '') AS warehouse_name,
        w.id AS warehouse_id,
        warehouse_name ILIKE '%' || CASE 
            WHEN SPLIT_PART(tag_name, '_', 1) = 'El' THEN SPLIT_PART(tag_name, '_', 2) 
            ELSE SPLIT_PART(tag_name, '_', 1) 
        END || '%' AS contains_flag
    FROM dynamic_tags dt
    JOIN dynamic_taggables dta ON dt.id = dta.dynamic_tag_id 
    CROSS JOIN warehouses w 
    WHERE dt.id > 3000
        AND dt.name LIKE '%QD_rets%'
        AND w.id IN (1, 236, 337, 8, 339, 170, 501, 401, 703, 632, 797, 962)
        AND contains_flag = 'true'
),

qd_config AS (
    SELECT * 
    FROM (
        SELECT 
            product_id,
            start_at,
            end_at,
            packing_unit_id,
            id AS qd_id,
            qd.warehouse_id,
            MAX(CASE WHEN tier = 1 THEN quantity END) AS tier_1_qty,
            MAX(CASE WHEN tier = 1 THEN discount_percentage END) AS tier_1_discount_pct,
            MAX(CASE WHEN tier = 2 THEN quantity END) AS tier_2_qty,
            MAX(CASE WHEN tier = 2 THEN discount_percentage END) AS tier_2_discount_pct,
            MAX(CASE WHEN tier = 3 THEN quantity END) AS tier_3_qty,
            MAX(CASE WHEN tier = 3 THEN discount_percentage END) AS tier_3_discount_pct
        FROM (
            SELECT 
                qd.id,
                qdv.product_id,
                qdv.packing_unit_id,
                qdv.quantity,
                qdv.discount_percentage,
                qd.dynamic_tag_id,
                qd.start_at,
                qd.end_at,
                ROW_NUMBER() OVER (
                    PARTITION BY qdv.product_id, qdv.packing_unit_id, qd.id 
                    ORDER BY qdv.quantity
                ) AS tier
            FROM quantity_discounts qd 
            JOIN quantity_discount_values qdv ON qd.id = qdv.quantity_discount_id 
            WHERE CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1 
                  BETWEEN qd.start_at::DATE AND qd.end_at::DATE
                AND qd.start_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 5
        ) qd_tiers
        JOIN qd_det qd ON qd.tag_id = qd_tiers.dynamic_tag_id
        GROUP BY ALL
    )
    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id, packing_unit_id, warehouse_id ORDER BY start_at DESC) = 1
),

-- Get all sales from yesterday
yesterday_sales AS (
    SELECT 
        COALESCE(pwh.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        so.retailer_id,
        pso.packing_unit_id,
        pso.purchased_item_count AS qty,
        pso.total_price AS nmv,
        pso.item_price / pso.basic_unit_count AS unit_price,
        pso.ITEM_DISCOUNT_VALUE AS sku_discount_per_unit,
        pso.ITEM_QUANTITY_DISCOUNT_VALUE AS qty_discount_per_unit,
        pso.ITEM_DISCOUNT_VALUE * pso.purchased_item_count AS sku_discount_total,
        pso.ITEM_QUANTITY_DISCOUNT_VALUE * pso.purchased_item_count AS qty_discount_total,
        qd.tier_1_qty,
        qd.tier_2_qty,
        qd.tier_3_qty,
        qd.tier_1_discount_pct,
        qd.tier_2_discount_pct,
        qd.tier_3_discount_pct,
        -- Determine tier used
        CASE 
            WHEN pso.ITEM_QUANTITY_DISCOUNT_VALUE = 0 OR qd.tier_1_qty IS NULL THEN 'Base'
            WHEN qd.tier_3_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_3_qty THEN 'Tier 3'
            WHEN qd.tier_2_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_2_qty THEN 'Tier 2'
            WHEN qd.tier_1_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_1_qty THEN 'Tier 1'
            ELSE 'Base'
        END AS tier_used
    FROM product_sales_order pso
    LEFT JOIN parent_whs pwh ON pwh.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    LEFT JOIN qd_config qd 
        ON qd.product_id = pso.product_id 
        AND qd.packing_unit_id = pso.packing_unit_id
        AND qd.warehouse_id = COALESCE(pwh.parent_id, pso.warehouse_id)
    WHERE so.created_at::DATE = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
)

SELECT 
    warehouse_id,
    product_id,
    SUM(nmv) AS total_nmv,
    SUM(CASE WHEN sku_discount_per_unit > 0 THEN nmv ELSE 0 END) AS sku_discount_nmv,
    SUM(CASE WHEN qty_discount_per_unit > 0 THEN nmv ELSE 0 END) AS qty_discount_nmv,
    SUM(CASE WHEN tier_used = 'Tier 1' THEN nmv ELSE 0 END) AS tier1_nmv,
    SUM(CASE WHEN tier_used = 'Tier 2' THEN nmv ELSE 0 END) AS tier2_nmv,
    SUM(CASE WHEN tier_used = 'Tier 3' THEN nmv ELSE 0 END) AS tier3_nmv,
    -- Tier quantities and discount percentages (from the active QD config)
    MAX(tier_1_qty) AS tier_1_qty,
    MAX(tier_1_discount_pct) AS tier_1_discount_pct,
    MAX(tier_2_qty) AS tier_2_qty,
    MAX(tier_2_discount_pct) AS tier_2_discount_pct,
    MAX(tier_3_qty) AS tier_3_qty,
    MAX(tier_3_discount_pct) AS tier_3_discount_pct
FROM yesterday_sales
GROUP BY warehouse_id, product_id
HAVING SUM(nmv) > 0
ORDER BY total_nmv DESC
'''

# Execute yesterday discount query
print("Loading yesterday's discount analysis data...")
df_yesterday_discount = query_snowflake(YESTERDAY_DISCOUNT_QUERY)
df_yesterday_discount = convert_to_numeric(df_yesterday_discount)
print(f"Loaded {len(df_yesterday_discount)} SKU discount records from yesterday")

# Calculate contributions in Python
df_yesterday_discount['sku_discount_nmv_cntrb'] = (
    df_yesterday_discount['sku_discount_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)
df_yesterday_discount['qty_discount_nmv_cntrb'] = (
    df_yesterday_discount['qty_discount_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)
df_yesterday_discount['tier1_nmv_cntrb'] = (
    df_yesterday_discount['tier1_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)
df_yesterday_discount['tier2_nmv_cntrb'] = (
    df_yesterday_discount['tier2_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)
df_yesterday_discount['tier3_nmv_cntrb'] = (
    df_yesterday_discount['tier3_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)

# Summary
print(f"\n{'='*60}")
print(f"YESTERDAY'S DISCOUNT ANALYSIS SUMMARY")
print(f"{'='*60}")
print(f"\nTotal NMV yesterday: {df_yesterday_discount['total_nmv'].sum():,.0f}")
print(f"SKU Discount NMV: {df_yesterday_discount['sku_discount_nmv'].sum():,.0f}")
print(f"Quantity Discount NMV: {df_yesterday_discount['qty_discount_nmv'].sum():,.0f}")
print(f"\nNMV by Tier:")
print(f"  Tier 1: {df_yesterday_discount['tier1_nmv'].sum():,.0f}")
print(f"  Tier 2: {df_yesterday_discount['tier2_nmv'].sum():,.0f}")
print(f"  Tier 3: {df_yesterday_discount['tier3_nmv'].sum():,.0f}")

df_yesterday_discount.head(10)


Loading yesterday's discount analysis data...
Loaded 10024 SKU discount records from yesterday

YESTERDAY'S DISCOUNT ANALYSIS SUMMARY

Total NMV yesterday: 28,592,420
SKU Discount NMV: 4,510,478
Quantity Discount NMV: 2,539,413

NMV by Tier:
  Tier 1: 761,564
  Tier 2: 1,392,804
  Tier 3: 351,087


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,warehouse_id,product_id,total_nmv,sku_discount_nmv,qty_discount_nmv,tier1_nmv,tier2_nmv,tier3_nmv,tier_1_qty,tier_1_discount_pct,tier_2_qty,tier_2_discount_pct,tier_3_qty,tier_3_discount_pct,sku_discount_nmv_cntrb,qty_discount_nmv_cntrb,tier1_nmv_cntrb,tier2_nmv_cntrb,tier3_nmv_cntrb
0,1,7630,358065.25,170428.00,0.00,0.0,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,47.60,0.00,0.00,0.00,0.0
1,962,7630,257129.25,90107.00,0.00,0.0,0.00,0.0,8.0,2.22,17.0,5.00,NaN,NaN,35.04,0.00,0.00,0.00,0.0
2,1,130,176241.00,4042.00,44892.00,7014.5,3503.50,34374.0,4.0,0.54,7.0,1.04,34.0,1.61,2.29,25.47,3.98,1.99,19.5
3,1,151,148580.00,7177.50,0.00,0.0,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,4.83,0.00,0.00,0.00,0.0
4,962,151,122170.75,4532.50,0.00,0.0,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,3.71,0.00,0.00,0.00,0.0
5,703,434,107211.00,86788.75,0.00,0.0,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,80.95,0.00,0.00,0.00,0.0
6,962,589,102588.15,13050.00,38497.50,3262.5,34365.00,0.0,5.0,1.08,10.0,2.39,NaN,NaN,12.72,37.53,3.18,33.50,0.0
7,1,589,99191.70,17144.00,12615.00,3915.0,8700.00,0.0,5.0,0.46,10.0,1.15,79.0,1.61,17.28,12.72,3.95,8.77,0.0
8,1,2049,96196.25,6152.50,8321.00,5791.0,2530.00,0.0,9.0,0.50,20.0,1.00,303.0,1.61,6.40,8.65,6.02,2.63,0.0
9,962,5151,86718.25,1217.00,17092.25,0.0,17092.25,0.0,4.0,0.70,7.0,1.17,15.0,1.49,1.40,19.71,0.00,19.71,0.0


In [44]:
# =============================================================================
# Add Yesterday's Discount Analysis to pricing_with_discount (Contributions Only)
# =============================================================================

# Merge yesterday discount data with pricing_with_discount - contributions + tier config
pricing_with_discount = pricing_with_discount.merge(
    df_yesterday_discount[[
        'warehouse_id', 'product_id', 
        'sku_discount_nmv_cntrb', 'qty_discount_nmv_cntrb',
        'tier1_nmv_cntrb', 'tier2_nmv_cntrb', 'tier3_nmv_cntrb',
        'tier_1_qty', 'tier_1_discount_pct',
        'tier_2_qty', 'tier_2_discount_pct',
        'tier_3_qty', 'tier_3_discount_pct'
    ]].rename(columns={
        'sku_discount_nmv_cntrb': 'yesterday_sku_disc_cntrb',
        'qty_discount_nmv_cntrb': 'yesterday_qty_disc_cntrb',
        'tier1_nmv_cntrb': 'yesterday_t1_cntrb',
        'tier2_nmv_cntrb': 'yesterday_t2_cntrb',
        'tier3_nmv_cntrb': 'yesterday_t3_cntrb',
        'tier_1_qty': 'qd_tier_1_qty',
        'tier_1_discount_pct': 'qd_tier_1_disc_pct',
        'tier_2_qty': 'qd_tier_2_qty',
        'tier_2_discount_pct': 'qd_tier_2_disc_pct',
        'tier_3_qty': 'qd_tier_3_qty',
        'tier_3_discount_pct': 'qd_tier_3_disc_pct'
    }), 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill NaN for SKUs that had no sales yesterday
contrib_cols = [
    'yesterday_sku_disc_cntrb', 'yesterday_qty_disc_cntrb',
    'yesterday_t1_cntrb', 'yesterday_t2_cntrb', 'yesterday_t3_cntrb'
]
for col in contrib_cols:
    if col in pricing_with_discount.columns:
        pricing_with_discount[col] = pricing_with_discount[col].fillna(0)

# Fill NaN for QD tier config (0 means no tier configured)
qd_config_cols = [
    'qd_tier_1_qty', 'qd_tier_1_disc_pct',
    'qd_tier_2_qty', 'qd_tier_2_disc_pct',
    'qd_tier_3_qty', 'qd_tier_3_disc_pct'
]
for col in qd_config_cols:
    if col in pricing_with_discount.columns:
        pricing_with_discount[col] = pricing_with_discount[col].fillna(0)

print(f"Yesterday's discount contributions and QD tier config added!")
print(f"\nSKUs with discount data: {len(pricing_with_discount[pricing_with_discount['yesterday_sku_disc_cntrb'] > 0]) + len(pricing_with_discount[pricing_with_discount['yesterday_qty_disc_cntrb'] > 0])}")
print(f"SKUs with QD tier config: {len(pricing_with_discount[pricing_with_discount['qd_tier_1_qty'] > 0])}")
print(f"\nSample data with yesterday's discount contributions and QD tiers:")
pricing_with_discount[pricing_with_discount['qd_tier_1_qty'] > 0][
    ['product_id', 'warehouse_id', 'sku', 
     'yesterday_sku_disc_cntrb', 'yesterday_qty_disc_cntrb',
     'qd_tier_1_qty', 'qd_tier_1_disc_pct', 'qd_tier_2_qty', 'qd_tier_2_disc_pct', 'qd_tier_3_qty', 'qd_tier_3_disc_pct']
].head(15)


Yesterday's discount contributions and QD tier config added!

SKUs with discount data: 3140
SKUs with QD tier config: 3095

Sample data with yesterday's discount contributions and QD tiers:


,product_id,warehouse_id,sku,yesterday_sku_disc_cntrb,yesterday_qty_disc_cntrb,qd_tier_1_qty,qd_tier_1_disc_pct,qd_tier_2_qty,qd_tier_2_disc_pct,qd_tier_3_qty,qd_tier_3_disc_pct
6,2424,337,حفاضات جود كير مقاس 5 - 40 حفاضة,0.00,22.74,4.0,1.41,7.0,3.54,0.0,0.00
7,2424,8,حفاضات جود كير مقاس 5 - 40 حفاضة,0.00,0.00,4.0,0.50,7.0,1.00,14.0,1.71
24,958,339,لبن المراعى تريتس موز - 200 مل,0.00,50.25,4.0,1.33,7.0,4.45,0.0,0.00
25,958,170,لبن المراعى تريتس موز - 200 مل,0.00,0.00,4.0,1.33,7.0,4.45,0.0,0.00
31,3392,339,كلوركس كلور الوان برائحة الزهور - 2 لتر,0.00,0.00,4.0,0.60,7.0,1.00,15.0,1.61
34,9803,1,جبنة كيرى بالقشطة - 4 قطعه,0.00,0.00,8.0,0.50,13.0,1.00,447.0,1.61
35,9803,1,جبنة كيرى بالقشطة - 4 قطعه,0.00,0.00,8.0,0.50,13.0,1.00,447.0,1.61
36,9803,1,جبنة كيرى بالقشطة - 4 قطعه,0.00,0.00,8.0,0.50,13.0,1.00,447.0,1.61
40,1124,797,ستينج فراولة زجاج - 275 مل,0.00,24.98,4.0,0.63,7.0,1.22,87.0,1.77
41,151,501,لبن بخيره - 1 لتر,0.00,0.00,4.0,0.50,7.0,1.00,35.0,1.84


In [45]:
# =============================================================================
# Performance Benchmark Query
# Gets: Yesterday qty, Recent 7d qty, MTD qty, and P80 benchmarks (240 days)
# Uses materialized_views.stock_day_close for in-stock determination
# =============================================================================
PERFORMANCE_BENCHMARK_QUERY = f'''
-- =============================================================================
-- PERFORMANCE BENCHMARK QUERY WITH FALLBACK LOGIC
-- =============================================================================
-- This query provides performance benchmarks (P80) for SKUs with fallback logic:
-- 1. Primary: p80_daily_240d (calculated from 240-day history)
-- 2. Fallback 1: Median cat-brand quantity (per warehouse)
-- 3. Fallback 2: Median cat quantity (per warehouse)
-- 4. Final fallback: 5
-- =============================================================================

WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),

params AS (
    SELECT
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS today,
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1 AS yesterday,
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 240 AS history_start,
        DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) AS current_month_start,
        DAY(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) AS current_day_of_month
),

-- Product category and brand lookup
product_lookup AS (
    SELECT DISTINCT
        p.id AS product_id,
        b.name_ar AS brand,
        cat.name_ar AS cat
    FROM products p
    JOIN brands b ON b.id = p.brand_id
    JOIN categories cat ON cat.id = p.category_id
),

-- Daily sales aggregation (240 days) - includes qty and retailer count
daily_sales AS (
    SELECT
        COALESCE(pwh.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        so.created_at::DATE AS sale_date,
        SUM(pso.purchased_item_count * pso.basic_unit_count) AS daily_qty,
        COUNT(DISTINCT so.retailer_id) AS daily_retailers
    FROM product_sales_order pso
    LEFT JOIN parent_whs pwh ON pwh.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    CROSS JOIN params p
    WHERE so.created_at::DATE >= p.history_start
        AND so.created_at::DATE < p.today
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY COALESCE(pwh.parent_id, pso.warehouse_id), pso.product_id, so.created_at::DATE
),

-- Daily stock status using stock_day_close
-- In-stock = opening (prev day close) > 0 AND closing > 0
daily_stock AS (
    SELECT
        COALESCE(pwh.parent_id, sdc.warehouse_id) AS warehouse_id,
        sdc.product_id,
        sdc.TIMESTAMP::DATE AS stock_date,
        sdc.available_stock,
        LAG(sdc.available_stock, 1) OVER (
            PARTITION BY COALESCE(pwh.parent_id, sdc.warehouse_id), sdc.product_id 
            ORDER BY sdc.TIMESTAMP::DATE
        ) AS opening_stock,
        CASE 
            WHEN LAG(sdc.available_stock, 1) OVER (
                    PARTITION BY COALESCE(pwh.parent_id, sdc.warehouse_id), sdc.product_id ORDER BY sdc.TIMESTAMP::DATE
                 ) > 0 
                 AND sdc.available_stock > 0 
            THEN 1 
            ELSE 0 
        END AS in_stock_flag
    FROM materialized_views.stock_day_close sdc
    LEFT JOIN parent_whs pwh ON pwh.child_id = sdc.warehouse_id
    CROSS JOIN params p
    WHERE sdc.TIMESTAMP::DATE >= p.history_start - 1  -- Need one extra day for LAG
        AND sdc.TIMESTAMP::DATE < p.today
),

-- Combine sales with stock status
daily_with_stock AS (
    SELECT
        COALESCE(ds.warehouse_id, st.warehouse_id) AS warehouse_id,
        COALESCE(ds.product_id, st.product_id) AS product_id,
        COALESCE(ds.sale_date, st.stock_date) AS the_date,
        COALESCE(ds.daily_qty, 0) AS daily_qty,
        COALESCE(ds.daily_retailers, 0) AS daily_retailers,
        COALESCE(st.in_stock_flag, 0) AS in_stock_flag
    FROM daily_sales ds
    FULL OUTER JOIN daily_stock st 
        ON ds.warehouse_id = st.warehouse_id 
        AND ds.product_id = st.product_id 
        AND ds.sale_date = st.stock_date
    WHERE COALESCE(ds.sale_date, st.stock_date) >= (SELECT history_start FROM params)
),

-- Add product category and brand to daily_with_stock for fallback calculations
daily_with_stock_cat_brand AS (
    SELECT
        dws.*,
        pl.brand,
        pl.cat
    FROM daily_with_stock dws
    LEFT JOIN product_lookup pl ON pl.product_id = dws.product_id
),

-- Identify new SKUs: those with sales ONLY in current month (no historical sales before current month)
new_sku_identification AS (
    SELECT
        warehouse_id,
        product_id,
        CASE 
            WHEN MAX(CASE WHEN the_date < (SELECT current_month_start FROM params) THEN 1 ELSE 0 END) = 0
                 AND MAX(CASE WHEN the_date >= (SELECT current_month_start FROM params) AND the_date < (SELECT today FROM params) THEN 1 ELSE 0 END) = 1
            THEN 1 
            ELSE 0 
        END AS is_new_sku
    FROM daily_with_stock
    WHERE daily_qty > 0  -- Only consider days with actual sales
    GROUP BY warehouse_id, product_id
),

-- Calculate P80 benchmark (in-stock days only, 240 days, EXCLUDING last 7 days)
p80_daily_benchmark AS (
    SELECT
        warehouse_id,
        product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY daily_qty) AS p80_daily_240d,
        AVG(daily_qty) AS avg_daily_240d,
        STDDEV(daily_qty) AS std_daily_240d,
        COUNT(*) AS in_stock_days_240d
    FROM daily_with_stock
    CROSS JOIN params p
    WHERE in_stock_flag = 1
        AND the_date >= p.history_start
        AND the_date < p.today - 7  -- Exclude last 7 days from benchmark
    GROUP BY warehouse_id, product_id
),

-- Calculate median cat-brand quantity (fallback 1)
-- Same filters: 240 days, in-stock only, exclude last 7 days, per warehouse
cat_brand_median AS (
    SELECT
        dws.warehouse_id,
        dws.cat,
        dws.brand,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dws.daily_qty) AS median_cat_brand_qty
    FROM daily_with_stock_cat_brand dws
    CROSS JOIN params p
    WHERE dws.in_stock_flag = 1
        AND dws.the_date >= p.history_start
        AND dws.the_date < p.today - 7  -- Exclude last 7 days from benchmark
        AND dws.cat IS NOT NULL
        AND dws.brand IS NOT NULL
    GROUP BY dws.warehouse_id, dws.cat, dws.brand
),

-- Calculate median cat quantity (fallback 2)
-- Same filters: 240 days, in-stock only, exclude last 7 days, per warehouse
cat_median AS (
    SELECT
        dws.warehouse_id,
        dws.cat,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dws.daily_qty) AS median_cat_qty
    FROM daily_with_stock_cat_brand dws
    CROSS JOIN params p
    WHERE dws.in_stock_flag = 1
        AND dws.the_date >= p.history_start
        AND dws.the_date < p.today - 7  -- Exclude last 7 days from benchmark
        AND dws.cat IS NOT NULL
    GROUP BY dws.warehouse_id, dws.cat
),

-- Calculate P80 retailer benchmark (in-stock days only, 240 days, EXCLUDING last 7 days)
p70_retailer_benchmark AS (
    SELECT
        warehouse_id,
        product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY daily_retailers) AS p70_daily_retailers_240d,
        AVG(daily_retailers) AS avg_daily_retailers_240d,
        STDDEV(daily_retailers) AS std_daily_retailers_240d
    FROM daily_with_stock
    CROSS JOIN params p
    WHERE in_stock_flag = 1
        AND the_date >= p.history_start
        AND the_date < p.today - 7  -- Exclude last 7 days from benchmark
    GROUP BY warehouse_id, product_id
),

-- NEW: Calculate current-month-only benchmarks for new SKUs
-- Use previous days of current month (excluding yesterday and last 1-2 days for stability)
current_month_benchmark AS (
    SELECT
        dws.warehouse_id,
        dws.product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY dws.daily_qty) AS p80_daily_current_month,
        AVG(dws.daily_qty) AS avg_daily_current_month,
        STDDEV(dws.daily_qty) AS std_daily_current_month,
        COUNT(*) AS in_stock_days_current_month
    FROM daily_with_stock dws
    CROSS JOIN params p
    INNER JOIN new_sku_identification nsi 
        ON dws.warehouse_id = nsi.warehouse_id 
        AND dws.product_id = nsi.product_id
    WHERE dws.in_stock_flag = 1
        AND dws.the_date >= p.current_month_start
        AND dws.the_date < p.yesterday - 1  -- Exclude yesterday and day before for stability
        AND nsi.is_new_sku = 1
    GROUP BY dws.warehouse_id, dws.product_id
    HAVING COUNT(*) >= 2  -- Need at least 2 days of data
),

-- NEW: Calculate current-month retailer benchmark for new SKUs
current_month_retailer_benchmark AS (
    SELECT
        dws.warehouse_id,
        dws.product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY dws.daily_retailers) AS p70_daily_retailers_current_month,
        AVG(dws.daily_retailers) AS avg_daily_retailers_current_month,
        STDDEV(dws.daily_retailers) AS std_daily_retailers_current_month
    FROM daily_with_stock dws
    CROSS JOIN params p
    INNER JOIN new_sku_identification nsi 
        ON dws.warehouse_id = nsi.warehouse_id 
        AND dws.product_id = nsi.product_id
    WHERE dws.in_stock_flag = 1
        AND dws.the_date >= p.current_month_start
        AND dws.the_date < p.yesterday - 1  -- Exclude yesterday and day before
        AND nsi.is_new_sku = 1
    GROUP BY dws.warehouse_id, dws.product_id
    HAVING COUNT(*) >= 2
),

-- Calculate 7-day rolling SUM for P80 recent benchmark
rolling_7d AS (
    SELECT
        warehouse_id,
        product_id,
        the_date,
        SUM(daily_qty) OVER (
            PARTITION BY warehouse_id, product_id 
            ORDER BY the_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7d_sum,
        SUM(in_stock_flag) OVER (
            PARTITION BY warehouse_id, product_id 
            ORDER BY the_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS in_stock_days_7d
    FROM daily_with_stock
),

p80_7d_benchmark AS (
    SELECT
        warehouse_id,
        product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY rolling_7d_sum) AS p80_7d_rolling_240d
    FROM rolling_7d
    CROSS JOIN params p
    WHERE the_date >= p.history_start + 7  -- Need 7 days for rolling
        AND the_date < p.today - 7  -- Exclude last 7 days from benchmark
        AND in_stock_days_7d >= 4  -- At least 4 of 7 days in stock
    GROUP BY warehouse_id, product_id
),

-- NEW: Calculate 7-day rolling benchmark for new SKUs using current month data
current_month_7d_benchmark AS (
    SELECT
        r7d.warehouse_id,
        r7d.product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY r7d.rolling_7d_sum) AS p80_7d_rolling_current_month
    FROM rolling_7d r7d
    CROSS JOIN params p
    INNER JOIN new_sku_identification nsi 
        ON r7d.warehouse_id = nsi.warehouse_id 
        AND r7d.product_id = nsi.product_id
    WHERE r7d.the_date >= p.current_month_start + 7  -- Need 7 days for rolling
        AND r7d.the_date < p.yesterday - 1  -- Exclude last 1-2 days
        AND r7d.in_stock_days_7d >= 4  -- At least 4 of 7 days in stock
        AND nsi.is_new_sku = 1
    GROUP BY r7d.warehouse_id, r7d.product_id
),

-- MTD benchmark: P80 of same MTD period totals (last 12 months)
-- Sum all sales from day 1 to current day of month for each historical month
mtd_historical AS (
    SELECT
        dws.warehouse_id,
        dws.product_id,
        DATE_TRUNC('month', dws.the_date) AS period_month_start,
        SUM(dws.daily_qty) AS mtd_total_qty  -- Sum of all days from 1 to current_day_of_month
    FROM daily_with_stock dws
    CROSS JOIN params p
    WHERE DAY(dws.the_date) <= p.current_day_of_month  -- Only days up to current day of month
    GROUP BY dws.warehouse_id, dws.product_id, DATE_TRUNC('month', dws.the_date)
),

mtd_by_period AS (
    SELECT
        mh.warehouse_id,
        mh.product_id,
        mh.period_month_start,
        mh.mtd_total_qty AS mtd_qty_at_day  -- Total MTD qty for that month
    FROM mtd_historical mh
    CROSS JOIN params p
    WHERE mh.period_month_start >= DATEADD(month, -12, p.current_month_start)
        AND mh.period_month_start < p.current_month_start
),

p80_mtd_benchmark AS (
    SELECT
        warehouse_id,
        product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY mtd_qty_at_day) AS p80_mtd_12mo,
        AVG(mtd_qty_at_day) AS avg_mtd_12mo
    FROM mtd_by_period
    GROUP BY warehouse_id, product_id
    HAVING COUNT(*) >= 3  -- At least 3 months of data
),

-- Current period quantities
current_metrics AS (
    SELECT
        warehouse_id,
        product_id,
        -- Yesterday
        SUM(CASE WHEN the_date = (SELECT yesterday FROM params) THEN daily_qty ELSE 0 END) AS yesterday_qty,
        SUM(CASE WHEN the_date = (SELECT yesterday FROM params) THEN daily_retailers ELSE 0 END) AS yesterday_retailers,
        -- Yesterday in-stock flag (1 if in stock yesterday, 0 otherwise)
        MAX(CASE WHEN the_date = (SELECT yesterday FROM params) THEN in_stock_flag ELSE 0 END) AS yesterday_in_stock,
        -- Recent 7 days
        SUM(CASE WHEN the_date >= (SELECT today FROM params) - 7 AND the_date < (SELECT today FROM params) THEN daily_qty ELSE 0 END) AS recent_7d_qty,
        SUM(CASE WHEN the_date >= (SELECT today FROM params) - 7 AND the_date < (SELECT today FROM params) AND in_stock_flag = 1 THEN 1 ELSE 0 END) AS recent_7d_in_stock_days,
        -- MTD
        SUM(CASE WHEN the_date >= (SELECT current_month_start FROM params) AND the_date < (SELECT today FROM params) THEN daily_qty ELSE 0 END) AS mtd_qty,
        SUM(CASE WHEN the_date >= (SELECT current_month_start FROM params) AND the_date < (SELECT today FROM params) AND in_stock_flag = 1 THEN 1 ELSE 0 END) AS mtd_in_stock_days
    FROM daily_with_stock
    GROUP BY warehouse_id, product_id
),

-- Combined performance ratio calculation with dynamic weights and fallback logic
-- Priority: New SKU current month benchmarks > Historical benchmarks > Fallback (cat-brand/cat median/5)
combined_ratio_calc AS (
    SELECT
        cm.warehouse_id,
        cm.product_id,
        cm.yesterday_qty,
        cm.yesterday_retailers,
        cm.yesterday_in_stock,
        cm.recent_7d_qty,
        cm.recent_7d_in_stock_days,
        cm.mtd_qty,
        cm.mtd_in_stock_days,
        
        -- Check if this is a new SKU
        COALESCE(nsi.is_new_sku, 0) AS is_new_sku,
        
        -- Get product category and brand for fallback lookup
        pl.brand,
        pl.cat,
        
        -- Benchmark values: For new SKUs, use current month benchmarks first, then fallback
        -- For existing SKUs, use historical benchmarks with fallback
        COALESCE(
            -- If new SKU and has current month benchmark, use it
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
            -- Otherwise use historical benchmark
            pb.p80_daily_240d,
            -- Fallback chain
            cbm.median_cat_brand_qty,
            cm_median.median_cat_qty,
            5
        ) AS p80_daily_240d,
        
        -- Average, stddev, and in_stock_days: Use current month for new SKUs, historical for others
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.avg_daily_current_month ELSE NULL END,
            pb.avg_daily_240d,
            0
        ) AS avg_daily_240d,
        
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.std_daily_current_month ELSE NULL END,
            pb.std_daily_240d,
            0
        ) AS std_daily_240d,
        
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.in_stock_days_current_month ELSE NULL END,
            pb.in_stock_days_240d,
            0
        ) AS in_stock_days_240d,
        
        -- For 7d: Use current month for new SKUs, historical for others, then fallback
        COALESCE(
            -- If new SKU and has current month 7d benchmark, use it
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cm7d.p80_7d_rolling_current_month ELSE NULL END,
            -- Otherwise use historical 7d benchmark
            p7.p80_7d_rolling_240d,
            -- Fallback: use daily benchmark * 7
            COALESCE(
                CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                pb.p80_daily_240d,
                cbm.median_cat_brand_qty,
                cm_median.median_cat_qty,
                5
            ) * 7,
            35  -- 5 * 7 as final fallback
        ) AS p80_7d_sum_240d,
        
        -- For MTD: Use current month daily * days for new SKUs, historical MTD for others, then fallback
        COALESCE(
            -- If new SKU, use current month daily benchmark * days
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 
                 THEN cmb.p80_daily_current_month * (SELECT current_day_of_month FROM params)
                 ELSE NULL 
            END,
            -- Otherwise use historical MTD benchmark
            pm.p80_mtd_12mo,
            -- Fallback: use daily benchmark * days
            COALESCE(
                CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                pb.p80_daily_240d,
                cbm.median_cat_brand_qty,
                cm_median.median_cat_qty,
                5
            ) * (SELECT current_day_of_month FROM params),
            5 * (SELECT current_day_of_month FROM params)  -- Final fallback
        ) AS p80_mtd_12mo,
        
        -- Retailer benchmarks: Use current month for new SKUs, historical for others
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmrb.p70_daily_retailers_current_month ELSE NULL END,
            pr.p70_daily_retailers_240d,
            1
        ) AS p70_daily_retailers_240d,
        
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmrb.avg_daily_retailers_current_month ELSE NULL END,
            pr.avg_daily_retailers_240d,
            0
        ) AS avg_daily_retailers_240d,
        
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmrb.std_daily_retailers_current_month ELSE NULL END,
            pr.std_daily_retailers_240d,
            0
        ) AS std_daily_retailers_240d,
        
        -- Calculate base ratios (capped at 3) using benchmarks with new SKU priority
        LEAST(
            cm.yesterday_qty / NULLIF(
                COALESCE(
                    CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                    pb.p80_daily_240d,
                    cbm.median_cat_brand_qty,
                    cm_median.median_cat_qty,
                    5
                ), 0
            ), 3
        ) AS yesterday_ratio_capped,
        
        LEAST(
            cm.recent_7d_qty / NULLIF(
                COALESCE(
                    CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cm7d.p80_7d_rolling_current_month ELSE NULL END,
                    p7.p80_7d_rolling_240d,
                    COALESCE(
                        CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                        pb.p80_daily_240d,
                        cbm.median_cat_brand_qty,
                        cm_median.median_cat_qty,
                        5
                    ) * 7,
                    35
                ), 0
            ), 3
        ) AS recent_ratio_capped,
        
        LEAST(
            cm.mtd_qty / NULLIF(
                COALESCE(
                    CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 
                         THEN cmb.p80_daily_current_month * (SELECT current_day_of_month FROM params)
                         ELSE NULL 
                    END,
                    pm.p80_mtd_12mo,
                    COALESCE(
                        CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                        pb.p80_daily_240d,
                        cbm.median_cat_brand_qty,
                        cm_median.median_cat_qty,
                        5
                    ) * (SELECT current_day_of_month FROM params),
                    5 * (SELECT current_day_of_month FROM params)
                ), 0
            ), 3
        ) AS mtd_ratio_capped,
        
        -- In-stock percentages for each period
        cm.yesterday_in_stock AS yesterday_in_stock_pct,
        cm.recent_7d_in_stock_days / 7.0 AS recent_7d_in_stock_pct,
        cm.mtd_in_stock_days / NULLIF((SELECT current_day_of_month FROM params) - 1, 0) AS mtd_in_stock_pct,
        
        -- Raw weights (base: 20% yesterday, 40% recent 7d, 40% MTD) scaled by in-stock percentage
        0.2 * cm.yesterday_in_stock AS yesterday_raw_weight,
        0.4 * (cm.recent_7d_in_stock_days / 7.0) AS recent_7d_raw_weight,
        CASE 
            WHEN (SELECT current_day_of_month FROM params) >= 3 
            THEN 0.4 * COALESCE(cm.mtd_in_stock_days / NULLIF((SELECT current_day_of_month FROM params) - 1, 0), 0)
            ELSE 0 
        END AS mtd_raw_weight
        
    FROM current_metrics cm
    LEFT JOIN new_sku_identification nsi ON cm.warehouse_id = nsi.warehouse_id AND cm.product_id = nsi.product_id
    LEFT JOIN p80_daily_benchmark pb ON cm.warehouse_id = pb.warehouse_id AND cm.product_id = pb.product_id
    LEFT JOIN current_month_benchmark cmb ON cm.warehouse_id = cmb.warehouse_id AND cm.product_id = cmb.product_id
    LEFT JOIN p80_7d_benchmark p7 ON cm.warehouse_id = p7.warehouse_id AND cm.product_id = p7.product_id
    LEFT JOIN current_month_7d_benchmark cm7d ON cm.warehouse_id = cm7d.warehouse_id AND cm.product_id = cm7d.product_id
    LEFT JOIN p80_mtd_benchmark pm ON cm.warehouse_id = pm.warehouse_id AND cm.product_id = pm.product_id
    LEFT JOIN p70_retailer_benchmark pr ON cm.warehouse_id = pr.warehouse_id AND cm.product_id = pr.product_id
    LEFT JOIN current_month_retailer_benchmark cmrb ON cm.warehouse_id = cmrb.warehouse_id AND cm.product_id = cmrb.product_id
    LEFT JOIN product_lookup pl ON pl.product_id = cm.product_id
    LEFT JOIN cat_brand_median cbm ON cbm.warehouse_id = cm.warehouse_id 
        AND cbm.cat = pl.cat 
        AND cbm.brand = pl.brand
    LEFT JOIN cat_median cm_median ON cm_median.warehouse_id = cm.warehouse_id 
        AND cm_median.cat = pl.cat
),

-- Pre-calculate combined ratio to avoid repetition
final_with_combined AS (
    SELECT
        crc.*,
        -- Calculate combined performance ratio once
        CASE WHEN (crc.yesterday_raw_weight + crc.recent_7d_raw_weight + crc.mtd_raw_weight) > 0
        THEN (
            (crc.yesterday_raw_weight / (crc.yesterday_raw_weight + crc.recent_7d_raw_weight + crc.mtd_raw_weight)) * crc.yesterday_ratio_capped +
            (crc.recent_7d_raw_weight / (crc.yesterday_raw_weight + crc.recent_7d_raw_weight + crc.mtd_raw_weight)) * crc.recent_ratio_capped +
            (crc.mtd_raw_weight / (crc.yesterday_raw_weight + crc.recent_7d_raw_weight + crc.mtd_raw_weight)) * crc.mtd_ratio_capped
        )
        ELSE 0 END AS combined_perf_ratio
    FROM combined_ratio_calc crc
)

-- Final output (same columns as original, values adjusted when over-achieving)
SELECT
    f.warehouse_id,
    f.product_id,
    
    -- Current period quantities
    f.yesterday_qty,
    f.yesterday_retailers,
    f.recent_7d_qty,
    f.recent_7d_in_stock_days,
    f.mtd_qty,
    f.mtd_in_stock_days,
    
    -- Quantity Benchmarks (P80) - adjusted when over-achieving, with fallback already applied
    CASE WHEN f.combined_perf_ratio > 1.1 
         THEN ROUND(f.p80_daily_240d + 0.5 * f.std_daily_240d, 2)
         ELSE f.p80_daily_240d 
    END AS p80_daily_240d,
    f.avg_daily_240d,
    f.std_daily_240d,
    f.in_stock_days_240d,
    CASE WHEN f.combined_perf_ratio > 1.1 
         THEN ROUND(f.p80_7d_sum_240d + 0.5 * f.std_daily_240d * 7, 2)
         ELSE f.p80_7d_sum_240d 
    END AS p80_7d_sum_240d,
    CASE WHEN f.combined_perf_ratio > 1.1 
         THEN ROUND(f.p80_mtd_12mo + 0.5 * f.std_daily_240d * (SELECT current_day_of_month FROM params), 2)
         ELSE f.p80_mtd_12mo 
    END AS p80_mtd_12mo,
    
    -- Retailer Benchmarks (P80) - adjusted when over-achieving
    CASE WHEN f.combined_perf_ratio > 1.1 
         THEN ROUND(f.p70_daily_retailers_240d + 0.5 * f.std_daily_retailers_240d, 2)
         ELSE f.p70_daily_retailers_240d 
    END AS p70_daily_retailers_240d,
    f.avg_daily_retailers_240d,
    f.std_daily_retailers_240d,
    
    -- Performance ratios (adjusted when over-achieving)
    ROUND(f.yesterday_qty / NULLIF(
        CASE WHEN f.combined_perf_ratio > 1.1 
             THEN f.p80_daily_240d + 0.5 * f.std_daily_240d
             ELSE f.p80_daily_240d 
        END, 0), 2) AS yesterday_ratio,
    ROUND(f.recent_7d_qty / NULLIF(
        CASE WHEN f.combined_perf_ratio > 1.1 
             THEN f.p80_7d_sum_240d + 0.5 * f.std_daily_240d * 7
             ELSE f.p80_7d_sum_240d 
        END, 0), 2) AS recent_ratio,
    ROUND(f.mtd_qty / NULLIF(
        CASE WHEN f.combined_perf_ratio > 1.1 
             THEN f.p80_mtd_12mo + 0.5 * f.std_daily_240d * (SELECT current_day_of_month FROM params)
             ELSE f.p80_mtd_12mo 
        END, 0), 2) AS mtd_ratio,
    ROUND(f.yesterday_retailers / NULLIF(
        CASE WHEN f.combined_perf_ratio > 1.1 
             THEN f.p70_daily_retailers_240d + 0.5 * f.std_daily_retailers_240d
             ELSE f.p70_daily_retailers_240d 
        END, 0), 2) AS yesterday_retailer_ratio,
    
    -- Additional columns for visibility
    ROUND(f.combined_perf_ratio, 2) AS combined_perf_ratio,
    CASE WHEN f.combined_perf_ratio > 1.1 THEN 1 ELSE 0 END AS is_over_achiever,
    f.is_new_sku  -- Flag to identify new SKUs using current month benchmarks

FROM final_with_combined f
WHERE f.warehouse_id IN (1, 236, 337, 8, 339, 170, 501, 401, 703, 632, 797, 962);


'''

# Execute benchmark query
print("Loading performance benchmark data (this may take a moment due to 240-day history)...")
df_benchmarks = query_snowflake(PERFORMANCE_BENCHMARK_QUERY)
df_benchmarks = convert_to_numeric(df_benchmarks)
print(f"Loaded {len(df_benchmarks)} benchmark records")

# =============================================================================
# Apply Minimum Thresholds to Benchmark Values
# - Daily quantity benchmarks should not be below 5
# - Daily retailers benchmarks should not be less than 2
# This ensures performance calculations don't use unrealistic benchmarks
# =============================================================================
MIN_DAILY_QTY_BENCHMARK = 5
MIN_DAILY_RETAILERS_BENCHMARK = 5

# Apply minimums to daily benchmarks
df_benchmarks['p80_daily_240d'] = df_benchmarks['p80_daily_240d'].clip(lower=MIN_DAILY_QTY_BENCHMARK)
df_benchmarks['p70_daily_retailers_240d'] = df_benchmarks['p70_daily_retailers_240d'].clip(lower=MIN_DAILY_RETAILERS_BENCHMARK)

# Apply proportional minimums to interval-based benchmarks
df_benchmarks['p80_7d_sum_240d'] = df_benchmarks['p80_7d_sum_240d'].clip(lower=MIN_DAILY_QTY_BENCHMARK * 7)  # 35

# For MTD, calculate dynamic minimum based on days in current period
# mtd_in_stock_days represents how many days of data we have in current month
df_benchmarks['p80_mtd_12mo'] = df_benchmarks.apply(
    lambda row: max(row['p80_mtd_12mo'], MIN_DAILY_QTY_BENCHMARK * max(row['mtd_in_stock_days'], 1)),
    axis=1
)

print(f"Applied minimum thresholds: qty >= {MIN_DAILY_QTY_BENCHMARK}/day, retailers >= {MIN_DAILY_RETAILERS_BENCHMARK}/day")

# Preview
df_benchmarks


Loading performance benchmark data (this may take a moment due to 240-day history)...


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 309673 benchmark records
Applied minimum thresholds: qty >= 5/day, retailers >= 5/day


,warehouse_id,product_id,yesterday_qty,yesterday_retailers,recent_7d_qty,recent_7d_in_stock_days,mtd_qty,mtd_in_stock_days,p80_daily_240d,avg_daily_240d,...,p70_daily_retailers_240d,avg_daily_retailers_240d,std_daily_retailers_240d,yesterday_ratio,recent_ratio,mtd_ratio,yesterday_retailer_ratio,combined_perf_ratio,is_over_achiever,is_new_sku
0,962,14980,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,0.00,0.00,NaN,0.00,0.00,0,0
1,236,307,14,8,146,1,618,4,22.89,13.982456,...,15.67,9.140351,5.336661,0.61,0.91,1.03,0.51,1.19,1,0
2,236,8427,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,0.00,0.00,NaN,0.00,0.00,0,0
3,401,5758,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,NaN,NaN,NaN,0.00,0.00,0,0
4,339,18138,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,NaN,NaN,NaN,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309668,401,18291,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,NaN,NaN,NaN,0.00,0.00,0,0
309669,703,20062,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,0.00,0.00,NaN,0.00,0.00,0,0
309670,962,9351,0,0,0,0,0,0,5.00,0.090909,...,5.00,0.090909,0.301511,NaN,NaN,NaN,NaN,0.00,0,0
309671,337,7792,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,0.00,0.00,NaN,0.00,0.00,0,0


In [46]:
# =============================================================================
# Add Performance Benchmarks and Tags to pricing_with_discount
# =============================================================================

# Merge benchmark data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_benchmarks[[
        'warehouse_id', 'product_id',
        'yesterday_qty', 'yesterday_retailers', 'recent_7d_qty', 'recent_7d_in_stock_days', 'mtd_qty', 'mtd_in_stock_days',
        'p80_daily_240d', 'avg_daily_240d','std_daily_240d', 'in_stock_days_240d',
        'p80_7d_sum_240d', 'p80_mtd_12mo',
        'p70_daily_retailers_240d', 'avg_daily_retailers_240d', 'std_daily_retailers_240d',
        'yesterday_ratio', 'recent_ratio', 'mtd_ratio', 'yesterday_retailer_ratio'
    ]], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill NaN values
qty_cols = ['yesterday_qty', 'yesterday_retailers', 'recent_7d_qty', 'recent_7d_in_stock_days', 'mtd_qty', 'mtd_in_stock_days']
for col in qty_cols:
    pricing_with_discount[col] = pricing_with_discount[col].fillna(0)

benchmark_cols = ['p80_daily_240d', 'p80_7d_sum_240d', 'p80_mtd_12mo', 'p70_daily_retailers_240d']
for col in benchmark_cols:
    pricing_with_discount[col] = pricing_with_discount[col].fillna(5)  # Default to 1 to avoid division issues

ratio_cols = ['yesterday_ratio', 'recent_ratio', 'mtd_ratio', 'yesterday_retailer_ratio']
for col in ratio_cols:
    pricing_with_discount[col] = pricing_with_discount[col].fillna(0)

pricing_with_discount['avg_daily_240d'] = pricing_with_discount['avg_daily_240d'].fillna(0)
pricing_with_discount['in_stock_days_240d'] = pricing_with_discount['in_stock_days_240d'].fillna(0)
pricing_with_discount['avg_daily_retailers_240d'] = pricing_with_discount['avg_daily_retailers_240d'].fillna(0)
pricing_with_discount['std_daily_retailers_240d'] = pricing_with_discount['std_daily_retailers_240d'].fillna(0)

# =============================================================================
# Performance Tags - Classify each ratio
# =============================================================================
def get_performance_tag(ratio):
    """
    Classify performance based on ratio to benchmark
    On Track: 90% to 115% of benchmark
    Upper tiers: start from 115%
    Lower tiers: start from 90%
    """
    if pd.isna(ratio) or ratio == 0:
        return 'No Data'
    elif ratio >= 1.75:
        return 'Star Performer'      # 🌟 75%+ above benchmark
    elif ratio > 1.15:
        return 'Over Achiever'       # 🔥 15%+ above benchmark  
    elif ratio >= 0.90:
        return 'On Track'            # ✅ Meeting benchmark (90%-115%)
    elif ratio >= 0.70:
        return 'Underperforming'     # ⚠️ 10%-30% below benchmark
    elif ratio >= 0.40:
        return 'Struggling'          # 🔻 30%-60% below benchmark
    else:
        return 'Critical'            # 🚨 60%+ below benchmark

# Apply tags to each timeframe
pricing_with_discount['yesterday_status'] = pricing_with_discount['yesterday_ratio'].apply(get_performance_tag)
pricing_with_discount['recent_status'] = pricing_with_discount['recent_ratio'].apply(get_performance_tag)
pricing_with_discount['mtd_status'] = pricing_with_discount['mtd_ratio'].apply(get_performance_tag)

# =============================================================================
# Combined Performance Score (weighted average of ratios)
# Approach 2: Scale Weights by In-Stock Percentage
# =============================================================================

# Calculate days in month so far (excluding today)
days_in_month_so_far = max(TODAY.day - 1, 1)  # At least 1 to avoid division by zero

# Calculate in-stock percentages for each period
pricing_with_discount['yesterday_in_stock_pct'] = 1 - pricing_with_discount['oos_yesterday']
pricing_with_discount['recent_7d_in_stock_pct'] = pricing_with_discount['recent_7d_in_stock_days'] / 7
pricing_with_discount['mtd_in_stock_pct'] = pricing_with_discount['mtd_in_stock_days'] / days_in_month_so_far

# Base weights: Yesterday 20%, Recent 7d 40%, MTD 40%
# Scale by in-stock percentage
# NOTE: MTD weight = 0 for first 3 days of month (unreliable data)
MTD_RELIABLE_DAY = 3  # Only use MTD when day >= 3
pricing_with_discount['yesterday_raw_weight'] = 0.2 * pricing_with_discount['yesterday_in_stock_pct']
pricing_with_discount['recent_7d_raw_weight'] = 0.4 * pricing_with_discount['recent_7d_in_stock_pct']
pricing_with_discount['mtd_raw_weight'] = np.where(
    TODAY.day >= MTD_RELIABLE_DAY,
    0.4 * pricing_with_discount['mtd_in_stock_pct'],
    0  # Set MTD weight to 0 at start of month
)

# Total raw weight for normalization
pricing_with_discount['total_raw_weight'] = (
    pricing_with_discount['yesterday_raw_weight'] + 
    pricing_with_discount['recent_7d_raw_weight'] + 
    pricing_with_discount['mtd_raw_weight']
)

# Normalized weights (sum to 1)
pricing_with_discount['yesterday_norm_weight'] = np.where(
    pricing_with_discount['total_raw_weight'] > 0,
    pricing_with_discount['yesterday_raw_weight'] / pricing_with_discount['total_raw_weight'],
    0
)
pricing_with_discount['recent_7d_norm_weight'] = np.where(
    pricing_with_discount['total_raw_weight'] > 0,
    pricing_with_discount['recent_7d_raw_weight'] / pricing_with_discount['total_raw_weight'],
    0
)
pricing_with_discount['mtd_norm_weight'] = np.where(
    pricing_with_discount['total_raw_weight'] > 0,
    pricing_with_discount['mtd_raw_weight'] / pricing_with_discount['total_raw_weight'],
    0
)

# Combined performance ratio with dynamic weights based on in-stock days
pricing_with_discount['combined_perf_ratio'] = (
    pricing_with_discount['yesterday_norm_weight'] * pricing_with_discount['yesterday_ratio'].clip(upper=3) +
    pricing_with_discount['recent_7d_norm_weight'] * pricing_with_discount['recent_ratio'].clip(upper=3) +
    pricing_with_discount['mtd_norm_weight'] * pricing_with_discount['mtd_ratio'].clip(upper=3)
)

# Handle cases where total_raw_weight = 0 (completely OOS in all periods)
pricing_with_discount['combined_perf_ratio'] = pricing_with_discount['combined_perf_ratio'].fillna(0)

# Clean up intermediate columns (optional - keep for debugging)
weight_debug_cols = ['yesterday_in_stock_pct', 'recent_7d_in_stock_pct', 'mtd_in_stock_pct',
                     'yesterday_raw_weight', 'recent_7d_raw_weight', 'mtd_raw_weight', 'total_raw_weight',
                     'yesterday_norm_weight', 'recent_7d_norm_weight', 'mtd_norm_weight']
# Uncomment to drop: pricing_with_discount.drop(columns=weight_debug_cols, inplace=True)

print(f"\nDynamic weight calculation complete!")
print(f"Days in month so far: {days_in_month_so_far}")
print(f"\nSample of weight distributions:")
print(pricing_with_discount[pricing_with_discount['total_raw_weight'] > 0][
    ['product_id', 'warehouse_id', 'oos_yesterday', 'recent_7d_in_stock_days', 'mtd_in_stock_days',
     'yesterday_norm_weight', 'recent_7d_norm_weight', 'mtd_norm_weight', 'combined_perf_ratio']
].head(10))

pricing_with_discount['combined_status'] = pricing_with_discount['combined_perf_ratio'].apply(get_performance_tag)

# =============================================================================
# High Performer Flag (for immediate action consideration)
# =============================================================================
# Flag SKUs that are significantly over-achieving and may need action (price increase, etc.)
pricing_with_discount['high_performer_flag'] = np.where(
    (pricing_with_discount['yesterday_ratio'] >= 1.5) & 
    (pricing_with_discount['recent_ratio'] >= 1.3) &
    (pricing_with_discount['mtd_ratio'] >= 1.2),
    1, 0
)

# Star performer flag (exceptional - all metrics 2x+ benchmark)
pricing_with_discount['star_performer_flag'] = np.where(
    (pricing_with_discount['yesterday_ratio'] >= 2.0) & 
    (pricing_with_discount['recent_ratio'] >= 1.5) &
    (pricing_with_discount['mtd_ratio'] >= 1.5),
    1, 0
)

# =============================================================================
# Summary
# =============================================================================
print(f"Performance benchmarks added!")
print(f"\n{'='*60}")
print(f"PERFORMANCE STATUS DISTRIBUTION")
print(f"{'='*60}")

print(f"\nYesterday Status:")
print(pricing_with_discount['yesterday_status'].value_counts().to_string())

print(f"\nRecent 7d Status:")
print(pricing_with_discount['recent_status'].value_counts().to_string())

print(f"\nMTD Status:")
print(pricing_with_discount['mtd_status'].value_counts().to_string())

print(f"\nCombined Status:")
print(pricing_with_discount['combined_status'].value_counts().to_string())

print(f"\n{'='*60}")
print(f"HIGH PERFORMERS (Action Candidates)")
print(f"{'='*60}")
print(f"High Performers (flag=1): {len(pricing_with_discount[pricing_with_discount['high_performer_flag'] == 1])}")
print(f"Star Performers (flag=1): {len(pricing_with_discount[pricing_with_discount['star_performer_flag'] == 1])}")

# Show top performers
print(f"\nTop 15 Star Performers:")
pricing_with_discount[pricing_with_discount['star_performer_flag'] == 1].nlargest(15, 'combined_perf_ratio')[
    ['product_id', 'warehouse_id', 'sku', 
     'yesterday_ratio', 'recent_ratio', 'mtd_ratio', 'combined_perf_ratio',
     'yesterday_status', 'combined_status']
]



Dynamic weight calculation complete!
Days in month so far: 18

Sample of weight distributions:
    product_id  warehouse_id  oos_yesterday  recent_7d_in_stock_days  \
0         5811           339              0                        6   
1         5811           170              0                        7   
2        10594             1              0                        0   
3        10594             1              0                        0   
6         2424           337              0                        7   
7         2424             8              0                        7   
8         9153           703              1                        2   
9         2668           501              0                        7   
11        4076           339              0                        7   
12        4076           170              0                        7   

    mtd_in_stock_days  yesterday_norm_weight  recent_7d_norm_weight  \
0                  13               0.24

,product_id,warehouse_id,sku,yesterday_ratio,recent_ratio,mtd_ratio,combined_perf_ratio,yesterday_status,combined_status
20530,26394,1,ماجيك فينجرز قطع الكاكاو بالشوكولاته - 10 جنية,9.00,5.14,3.32,3.0,Star Performer,Star Performer
20531,26394,1,ماجيك فينجرز قطع الكاكاو بالشوكولاته - 10 جنية,9.00,5.14,3.32,3.0,Star Performer,Star Performer
52267,7005,401,اندومى لحمة بيف جامبو عرض 44 كيس - 100 جم,4.37,5.26,3.34,3.0,Star Performer,Star Performer
106860,19543,1,المراعي يوجو فراولة - 425 جم,7.11,4.67,3.38,3.0,Star Performer,Star Performer
107704,8285,703,صابون كامى كلاسيك -- 110 جم,20.22,3.68,3.12,3.0,Star Performer,Star Performer
1894,8284,1,صابون كامى حيوية - 110 جم,9.49,3.88,3.53,3.0,Star Performer,Star Performer
1895,8284,1,صابون كامى حيوية - 110 جم,9.49,3.88,3.53,3.0,Star Performer,Star Performer
19552,27223,236,بسكوت .بيمبو لارج شوكو ماكس 3 قطعه - 10 جنية,22.00,4.86,3.37,3.0,Star Performer,Star Performer
19553,27223,236,بسكوت .بيمبو لارج شوكو ماكس 3 قطعه - 10 جنية,22.00,4.86,3.37,3.0,Star Performer,Star Performer
21537,26126,401,أولويز ماكسى سميكة ناعمة بلمسة الألوڤيرا طوي...,4.91,10.23,4.93,3.0,Star Performer,Star Performer


In [47]:
# =============================================================================
# No NMV in Last 4 Months Flag
# Identifies SKUs that have not generated any NMV in the past 4 months (120 days)
# =============================================================================
NO_NMV_4M_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
nmv_last_4m AS (
    SELECT 
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        SUM(pso.total_price) AS total_nmv_4m
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
        AND so.created_at::DATE < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), pso.product_id
    HAVING SUM(pso.total_price) > 0
)
SELECT 
    warehouse_id,
    product_id,
    total_nmv_4m
FROM nmv_last_4m
'''

# Execute query
print("Loading SKUs with NMV in last 4 months...")
df_nmv_4m = query_snowflake(NO_NMV_4M_QUERY)
df_nmv_4m = convert_to_numeric(df_nmv_4m)
print(f"Found {len(df_nmv_4m)} SKU-warehouse combinations with NMV in last 4 months")

# Merge and create no_nmv_4m flag
pricing_with_discount = pricing_with_discount.merge(
    df_nmv_4m[['warehouse_id', 'product_id', 'total_nmv_4m']],
    on=['warehouse_id', 'product_id'],
    how='left'
)

# Flag SKUs with no NMV in last 4 months
# 1 = No NMV (should potentially be filtered), 0 = Has NMV
pricing_with_discount['no_nmv_4m'] = np.where(
    pricing_with_discount['total_nmv_4m'].isna() | (pricing_with_discount['total_nmv_4m'] == 0),
    1, 0
)

# Fill NaN for total_nmv_4m
pricing_with_discount['total_nmv_4m'] = pricing_with_discount['total_nmv_4m'].fillna(0)

print(f"\n{'='*60}")
print(f"NO NMV IN LAST 4 MONTHS ANALYSIS")
print(f"{'='*60}")
print(f"Total records: {len(pricing_with_discount)}")
print(f"SKUs with NO NMV in 4 months (no_nmv_4m=1): {len(pricing_with_discount[pricing_with_discount['no_nmv_4m'] == 1])}")
print(f"SKUs with NMV in 4 months (no_nmv_4m=0): {len(pricing_with_discount[pricing_with_discount['no_nmv_4m'] == 0])}")

# Show sample of SKUs with no NMV
print(f"\nSample SKUs with no NMV in last 4 months:")
pricing_with_discount[pricing_with_discount['no_nmv_4m'] == 1][
    ['product_id', 'warehouse_id', 'sku', 'stocks', 'in_stock_rr', 'zero_demand', 'no_nmv_4m']
].head(15)


Loading SKUs with NMV in last 4 months...


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Found 30438 SKU-warehouse combinations with NMV in last 4 months

NO NMV IN LAST 4 MONTHS ANALYSIS
Total records: 108435
SKUs with NO NMV in 4 months (no_nmv_4m=1): 70782
SKUs with NMV in 4 months (no_nmv_4m=0): 37653

Sample SKUs with no NMV in last 4 months:


,product_id,warehouse_id,sku,stocks,in_stock_rr,zero_demand,no_nmv_4m
4,6224,501,بانيه كوكى توابل شرقية بارد 20 قطعة - 1 كجم,0,0.0,0,1
5,24709,401,بلـيزو سوداني محشو بالشوكولاتة مغطي بالسوداني ...,0,0.0,0,1
10,9288,501,مناديل بابيا نايلون 2 طبقة - 500 منديل,0,0.0,0,1
13,10780,632,بطارية ريموت افيريدى ازرق - 20 حجر,0,0.0,0,1
16,11719,401,ماجيك فينجرز ويفر كاكاو محشو بكريمة الشوكولاتة...,0,0.0,0,1
18,11502,703,مكرونة الملكة ملوكى اسباجتى - 400 جم,0,0.0,0,1
19,11502,703,مكرونة الملكة ملوكى اسباجتى - 400 جم,0,0.0,0,1
27,6799,797,ستربس كوكى بارد - 400 جم,0,0.0,0,1
33,10483,401,كفته بقري توابل شرقيه - 900 جم,0,0.0,0,1
39,3047,632,شوكولاتة جلاكسى فلوتس سنجل - 11.25 جم,0,0.0,0,1


In [48]:
# =============================================================================
# Normal Refill Query - Avg qty & stddev for frequent retailers (last 120 days)
# Frequent retailer definition based on ABC classification (from existing dataframe):
#   - Class A: bought 4+ times
#   - Class B: bought 3+ times
#   - Class C: bought 2+ times
# =============================================================================
NORMAL_REFILL_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
params AS (
    SELECT 
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS today,
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120 AS history_start
),

-- Get retailer order counts per product-warehouse (last 120 days)
retailer_orders AS (
    SELECT 
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        so.retailer_id,
        COUNT(DISTINCT so.id) AS order_count
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    CROSS JOIN params p
    WHERE so.created_at::DATE >= p.history_start
        AND so.created_at::DATE < p.today
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), pso.product_id, so.retailer_id
),

-- Get individual order quantities per retailer
order_quantities AS (
    SELECT 
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        so.retailer_id,
        so.id AS order_id,
        SUM(pso.purchased_item_count * pso.basic_unit_count) AS order_qty
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    CROSS JOIN params p
    WHERE so.created_at::DATE >= p.history_start
        AND so.created_at::DATE < p.today
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), pso.product_id, so.retailer_id, so.id
)

-- Return retailer-level data with order counts for Python filtering
SELECT 
    oq.warehouse_id,
    oq.product_id,
    oq.retailer_id,
    ro.order_count,
    oq.order_id,
    oq.order_qty
FROM order_quantities oq
JOIN retailer_orders ro 
    ON ro.warehouse_id = oq.warehouse_id 
    AND ro.product_id = oq.product_id 
    AND ro.retailer_id = oq.retailer_id
'''

# Execute normal refill query
print("Loading retailer order data for normal refill calculation (last 120 days)...")
df_retailer_orders = query_snowflake(NORMAL_REFILL_QUERY)
df_retailer_orders = convert_to_numeric(df_retailer_orders)
print(f"Loaded {len(df_retailer_orders)} retailer order records")

# Get ABC classification from existing dataframe
abc_mapping = pricing_with_discount[['warehouse_id', 'product_id', 'abc_class']].drop_duplicates()
print(f"ABC classification mapping: {len(abc_mapping)} product-warehouse combinations")

# Merge ABC classification into retailer orders
df_retailer_orders = df_retailer_orders.merge(
    abc_mapping,
    on=['warehouse_id', 'product_id'],
    how='inner'
)
print(f"Records after ABC merge: {len(df_retailer_orders)}")

# Filter frequent retailers based on ABC class thresholds
# Class A: 4+ orders, Class B: 3+ orders, Class C: 2+ orders
df_frequent = df_retailer_orders[
    ((df_retailer_orders['abc_class'] == 'A') & (df_retailer_orders['order_count'] >= 2)) |
    ((df_retailer_orders['abc_class'] == 'B') & (df_retailer_orders['order_count'] >= 2)) |
    ((df_retailer_orders['abc_class'] == 'C') & (df_retailer_orders['order_count'] >= 2))
].copy()
print(f"Records from frequent retailers: {len(df_frequent)}")

# Calculate normal_refill (avg qty) and refill_stddev per product-warehouse
df_normal_refill = df_frequent.groupby(['warehouse_id', 'product_id']).agg(
    frequent_retailer_count=('retailer_id', 'nunique'),
    frequent_order_count=('order_id', 'nunique'),
    normal_refill=('order_qty', 'mean'),
    refill_stddev=('order_qty', 'std')
).reset_index()

# Round values and fill NaN stddev (when only 1 order)
df_normal_refill['normal_refill'] = df_normal_refill['normal_refill'].round(2)
df_normal_refill['refill_stddev'] = df_normal_refill['refill_stddev'].fillna(0).round(2)

# Filter to products with at least 2 orders for meaningful stats
df_normal_refill = df_normal_refill[df_normal_refill['frequent_order_count'] >= 2]
print(f"Final normal refill records (min 2 orders): {len(df_normal_refill)}")

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_normal_refill[['warehouse_id', 'product_id', 'frequent_retailer_count', 
                      'frequent_order_count', 'normal_refill', 'refill_stddev']],
    on=['warehouse_id', 'product_id'],
    how='left'
)

# Fill NaN values
pricing_with_discount['frequent_retailer_count'] = pricing_with_discount['frequent_retailer_count'].fillna(0)
pricing_with_discount['frequent_order_count'] = pricing_with_discount['frequent_order_count'].fillna(0)
pricing_with_discount['normal_refill'] = pricing_with_discount['normal_refill'].fillna(0)
pricing_with_discount['refill_stddev'] = pricing_with_discount['refill_stddev'].fillna(0)

print(f"\n{'='*60}")
print(f"NORMAL REFILL ANALYSIS (Frequent Retailers - 120 days)")
print(f"{'='*60}")
print(f"Records with normal_refill data: {len(pricing_with_discount[pricing_with_discount['normal_refill'] > 0])}")
print(f"Records without normal_refill data: {len(pricing_with_discount[pricing_with_discount['normal_refill'] == 0])}")
print(f"\nNormal refill distribution:")
print(pricing_with_discount[pricing_with_discount['normal_refill'] > 0]['normal_refill'].describe())
print(f"\nSample data:")
pricing_with_discount[pricing_with_discount['normal_refill'] > 0][
    ['product_id', 'warehouse_id', 'sku', 'abc_class', 'frequent_retailer_count', 
     'frequent_order_count', 'normal_refill', 'refill_stddev', 'in_stock_rr']
].head(15)


Loading retailer order data for normal refill calculation (last 120 days)...


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 4990714 retailer order records
ABC classification mapping: 88626 product-warehouse combinations
Records after ABC merge: 4869412
Records from frequent retailers: 2655961
Final normal refill records (min 2 orders): 23679

NORMAL REFILL ANALYSIS (Frequent Retailers - 120 days)
Records with normal_refill data: 30413
Records without normal_refill data: 78022

Normal refill distribution:
count    30413.000000
mean         3.427859
std          4.380413
min          1.000000
25%          1.400000
50%          2.060000
75%          3.620000
max        135.060000
Name: normal_refill, dtype: float64

Sample data:


,product_id,warehouse_id,sku,abc_class,frequent_retailer_count,frequent_order_count,normal_refill,refill_stddev,in_stock_rr
0,5811,339,بسكويت بيمبو بندق - 5 جنية,B,61.0,161.0,3.91,3.93,34.0
1,5811,170,بسكويت بيمبو بندق - 5 جنية,B,53.0,154.0,2.99,2.59,12.0
2,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,C,50.0,141.0,1.57,1.28,4.0
3,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,C,50.0,141.0,1.57,1.28,4.0
6,2424,337,حفاضات جود كير مقاس 5 - 40 حفاضة,B,182.0,628.0,4.82,5.97,64.0
7,2424,8,حفاضات جود كير مقاس 5 - 40 حفاضة,B,232.0,812.0,10.01,21.96,169.0
8,9153,703,زيت شعر فيانسيه مغذى باللوز - 175 مل,C,3.0,6.0,4.50,2.35,4.0
11,4076,339,شوكولاتة جلاكسى سادة - 18 جم,B,73.0,185.0,1.76,1.37,11.0
12,4076,170,شوكولاتة جلاكسى سادة - 18 جم,B,69.0,185.0,1.78,1.54,8.0
14,1700,1,مناديل فاميليا سحب 2 طبقة - 500 منديل,C,2.0,6.0,2.83,1.83,3.0


In [49]:
# =============================================================================
# Live Cart Rules Query - Get current cart rules from the system
# Merges on product_id and cohort_id
# =============================================================================
LIVE_CART_RULES_QUERY = f'''
SELECT 
    cppu.cohort_id,
    pup.product_id,
    pup.packing_unit_id,
    pup.basic_unit_count,
    COALESCE(cppu.MAX_PER_SALES_ORDER, cppu2.MAX_PER_SALES_ORDER) AS current_cart_rule
FROM COHORT_PRODUCT_PACKING_UNITS cppu 
JOIN PACKING_UNIT_PRODUCTS pup ON cppu.PRODUCT_PACKING_UNIT_ID = pup.id 
JOIN cohorts c ON c.id = cppu.cohort_id
LEFT JOIN COHORT_PRODUCT_PACKING_UNITS cppu2 
    ON cppu.PRODUCT_PACKING_UNIT_ID = cppu2.PRODUCT_PACKING_UNIT_ID 
    AND cppu2.cohort_id = c.FALLBACK_COHORT_ID
WHERE cppu.cohort_id IN ({','.join(map(str, COHORT_IDS))})
'''

# Execute live cart rules query
print("Loading live cart rules...")
df_cart_rules = query_snowflake(LIVE_CART_RULES_QUERY)
df_cart_rules = convert_to_numeric(df_cart_rules)
print(f"Loaded {len(df_cart_rules)} cart rule records")

# Aggregate to product-cohort level (take the cart rule for basic unit, or min if multiple)
# Filter to basic unit (packing_unit_id where basic_unit_count = 1) for simpler merging
df_cart_rules_basic = df_cart_rules[df_cart_rules['basic_unit_count'] == 1].copy()
print(f"Basic unit cart rules: {len(df_cart_rules_basic)}")

# If no basic unit, take the minimum cart rule per product-cohort
df_cart_rules_agg = df_cart_rules.groupby(['cohort_id', 'product_id']).agg(
    current_cart_rule=('current_cart_rule', 'min')
).reset_index()

# Prefer basic unit cart rule, fallback to aggregated
df_cart_rules_final = df_cart_rules_basic[['cohort_id', 'product_id', 'current_cart_rule']].drop_duplicates()
df_cart_rules_final = df_cart_rules_final.merge(
    df_cart_rules_agg[['cohort_id', 'product_id', 'current_cart_rule']].rename(columns={'current_cart_rule': 'cart_rule_agg'}),
    on=['cohort_id', 'product_id'],
    how='outer'
)
df_cart_rules_final['current_cart_rule'] = df_cart_rules_final['current_cart_rule'].fillna(df_cart_rules_final['cart_rule_agg'])
df_cart_rules_final = df_cart_rules_final[['cohort_id', 'product_id', 'current_cart_rule']].drop_duplicates()
print(f"Final cart rules (product-cohort level): {len(df_cart_rules_final)}")

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_cart_rules_final,
    on=['cohort_id', 'product_id'],
    how='left'
)

# Fill NaN cart rules with 0 (no cart rule set)
pricing_with_discount['current_cart_rule'] = pricing_with_discount['current_cart_rule'].fillna(0)

print(f"\n{'='*60}")
print(f"LIVE CART RULES ANALYSIS")
print(f"{'='*60}")
print(f"Records with cart rule > 0: {len(pricing_with_discount[pricing_with_discount['current_cart_rule'] > 0])}")
print(f"Records without cart rule: {len(pricing_with_discount[pricing_with_discount['current_cart_rule'] == 0])}")
print(f"\nCart rule distribution:")
print(pricing_with_discount[pricing_with_discount['current_cart_rule'] > 0]['current_cart_rule'].describe())
print(f"\nSample data with cart rules:")
pricing_with_discount[pricing_with_discount['current_cart_rule'] > 0][
    ['product_id', 'cohort_id', 'warehouse_id', 'sku', 'current_price', 'current_cart_rule', 'in_stock_rr']
].head(15)


Loading live cart rules...


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 113112 cart rule records
Basic unit cart rules: 74690
Final cart rules (product-cohort level): 74681

LIVE CART RULES ANALYSIS
Records with cart rule > 0: 108474
Records without cart rule: 0

Cart rule distribution:
count    108474.000000
mean         49.484098
std         533.111089
min           2.000000
25%          10.000000
50%          12.000000
75%          25.000000
max       10000.000000
Name: current_cart_rule, dtype: float64

Sample data with cart rules:


,product_id,cohort_id,warehouse_id,sku,current_price,current_cart_rule,in_stock_rr
0,5811,704,339,بسكويت بيمبو بندق - 5 جنية,54.25,15,34.0
1,5811,704,170,بسكويت بيمبو بندق - 5 جنية,54.25,15,12.0
2,10594,700,1,كورونا شوكولاتة داكنة 72%- 45 جنية,240.00,12,4.0
3,10594,700,1,كورونا شوكولاتة داكنة 72%- 45 جنية,240.00,12,4.0
4,6224,1124,501,بانيه كوكى توابل شرقية بارد 20 قطعة - 1 كجم,127.00,10,0.0
5,24709,1126,401,بلـيزو سوداني محشو بالشوكولاتة مغطي بالسوداني ...,48.00,10,0.0
6,2424,703,337,حفاضات جود كير مقاس 5 - 40 حفاضة,178.25,31,64.0
7,2424,703,8,حفاضات جود كير مقاس 5 - 40 حفاضة,178.25,31,169.0
8,9153,1123,703,زيت شعر فيانسيه مغذى باللوز - 175 مل,30.00,12,4.0
9,2668,1124,501,كلوركس جل انتعاش الليمون - 750 مل,497.00,10,0.0


In [50]:
# =============================================================================
# Commercial Constraint Minimum Price Query
# Gets the minimum price constraints from finance.minimum_prices
# =============================================================================
COMMERCIAL_MIN_PRICE_QUERY = f'''
WITH to_remove AS (
    SELECT 
        check_date AS start_date,
        (check_date + INTERVAL '1 month') + 6 AS end_date 
    FROM (
        SELECT 
            CASE 
                WHEN DATE_PART('day', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) < 7 
                THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
                ELSE DATE_FROM_PARTS(
                    YEAR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE), 
                    MONTH(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE), 
                    1
                )  
            END AS check_date
    )
),
region_mapping as(
select *
from (
values
('Greater Cairo', 'Cairo'),
('Greater Cairo', 'Giza')

)x(region,main_region)

)

SELECT  
    sku_id AS product_id,
    sku,
    brand AS comm_brand,
    cat AS comm_cat,
    region,
    created_at AS comm_created_at,
    min_price AS commercial_min_price
FROM (
    SELECT 
        product_id AS sku_id,
        product_name AS sku,
        brand,
        category AS cat,
        coalesce(rm.main_region,mp.region) as region,
        min_price,
        created_at,
        MAX(created_at) OVER (PARTITION BY product_id, mp.region) AS latest_date
    FROM finance.minimum_prices mp
	left join region_mapping rm on mp.region = rm.region
    WHERE is_deleted = 'false'
        AND created_at BETWEEN (SELECT start_date FROM to_remove) AND (SELECT end_date FROM to_remove)
) comm
WHERE created_at = latest_date
'''

# Execute commercial min price query
print("Loading commercial minimum price constraints...")
df_commercial_min = query_snowflake(COMMERCIAL_MIN_PRICE_QUERY)
df_commercial_min = convert_to_numeric(df_commercial_min)
print(f"Loaded {len(df_commercial_min)} commercial min price records")

# Merge with pricing_with_discount on product_id and region
pricing_with_discount = pricing_with_discount.merge(
    df_commercial_min[['product_id', 'region', 'commercial_min_price']],
    on=['product_id', 'region'],
    how='left'
)

# Fill NaN with 0 (no commercial constraint)
pricing_with_discount['commercial_min_price'] = pricing_with_discount['commercial_min_price'].fillna(0)

print(f"\n{'='*60}")
print(f"COMMERCIAL MINIMUM PRICE CONSTRAINTS")
print(f"{'='*60}")
print(f"Records with commercial min price: {len(pricing_with_discount[pricing_with_discount['commercial_min_price'] > 0])}")
print(f"Records without commercial min price: {len(pricing_with_discount[pricing_with_discount['commercial_min_price'] == 0])}")
print(f"\nCommercial min price distribution:")
print(pricing_with_discount[pricing_with_discount['commercial_min_price'] > 0]['commercial_min_price'].describe())
print(f"\nSample data with commercial constraints:")
pricing_with_discount[pricing_with_discount['commercial_min_price'] > 0][
    ['product_id', 'region', 'warehouse_id', 'sku', 'current_price', 'commercial_min_price', 'price_after_discount']
].head(15)


Loading commercial minimum price constraints...
Loaded 767 commercial min price records

COMMERCIAL MINIMUM PRICE CONSTRAINTS
Records with commercial min price: 1783
Records without commercial min price: 106691

Commercial min price distribution:
count    1783.000000
mean      401.395922
std       336.731675
min        29.000000
25%       205.000000
50%       274.078350
75%       543.200000
max      2295.699000
Name: commercial_min_price, dtype: float64

Sample data with commercial constraints:


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,product_id,region,warehouse_id,sku,current_price,commercial_min_price,price_after_discount
2,10594,Cairo,1,كورونا شوكولاتة داكنة 72%- 45 جنية,240.00,240.000000,240.0000
3,10594,Cairo,1,كورونا شوكولاتة داكنة 72%- 45 جنية,240.00,240.000000,240.0000
17,25718,Upper Egypt,632,صلصة رشيدي الميزان برطمان - 300 جم,300.75,300.675000,300.7500
134,10616,Delta East,339,حلو الشام سحلب كيس - 150 جم,403.25,403.325999,403.2500
135,10616,Delta East,170,حلو الشام سحلب كيس - 150 جم,403.25,403.325999,403.2500
138,3849,Upper Egypt,703,دومتي عصير تفاح- 235 مل,188.00,169.750000,188.0000
287,19987,Upper Egypt,632,دومتي جبنة فلمنك - 500 جم,480.25,480.150000,480.2500
480,8651,Upper Egypt,501,شويبس جولد خوخ - 950 مل,160.00,152.000000,160.0000
946,10670,Giza,236,بسكاتو كريمة كسترد - 10 جنية,84.00,84.000000,80.9928
947,10670,Giza,236,بسكاتو كريمة كسترد - 10 جنية,84.00,84.000000,80.9928


In [51]:
# =============================================================================
# Active SKU Discount Query - Get current SKU discount percentage per warehouse
# =============================================================================
ACTIVE_SKU_DISCOUNT_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
active_sku_discount AS ( 
    SELECT 
        x.id AS sku_discount_id,
        retailer_id,
        product_id,
        packing_unit_id,
        DISCOUNT_PERCENTAGE,
        start_at,
        end_at 
    FROM (
        SELECT 
            sd.*,
            f.value::INT AS retailer_id 
        FROM SKU_DISCOUNTS sd,
        LATERAL FLATTEN(
            input => SPLIT(
                REPLACE(REPLACE(REPLACE(sd.retailer_ids, '{{', ''), '}}', ''), '"', ''), 
                ','
            )
        ) f
        WHERE start_at::DATE <= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
            AND end_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
    ) x 
    JOIN SKU_DISCOUNT_VALUES sdv ON x.id = sdv.sku_discount_id
    WHERE name_en = 'Special Discounts'
    QUALIFY MAX(start_at) OVER (PARTITION BY retailer_id, product_id, packing_unit_id) = start_at 
)

SELECT 
    product_id, 
    COALESCE(pwh.parent_id, sub.warehouse_id) AS warehouse_id,
    AVG(DISCOUNT_PERCENTAGE) AS active_sku_disc_pct 
FROM (
    SELECT 
        asd.*,
        warehouse_id 
    FROM active_sku_discount asd 
    JOIN materialized_views.retailer_polygon rp ON rp.retailer_id = asd.retailer_id
    JOIN WAREHOUSE_DISPATCHING_RULES wdr ON wdr.product_id = asd.product_id
    JOIN DISPATCHING_POLYGONS dp ON dp.id = wdr.DISPATCHING_POLYGON_ID AND dp.district_id = rp.district_id
) sub
LEFT JOIN parent_whs pwh ON pwh.child_id = sub.warehouse_id
GROUP BY ALL
'''

# Execute active SKU discount query
print("Loading active SKU discount data...")
df_active_sku_disc = query_snowflake(ACTIVE_SKU_DISCOUNT_QUERY)
df_active_sku_disc = convert_to_numeric(df_active_sku_disc)
print(f"Loaded {len(df_active_sku_disc)} active SKU discount records")

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_active_sku_disc[['product_id', 'warehouse_id', 'active_sku_disc_pct']],
    on=['product_id', 'warehouse_id'],
    how='left'
)

# Fill NaN with 0 (no active SKU discount)
pricing_with_discount['active_sku_disc_pct'] = pricing_with_discount['active_sku_disc_pct'].fillna(0)

print(f"\n{'='*60}")
print(f"ACTIVE SKU DISCOUNT ANALYSIS")
print(f"{'='*60}")
print(f"Records with active SKU discount: {len(pricing_with_discount[pricing_with_discount['active_sku_disc_pct'] > 0])}")
print(f"Records without active SKU discount: {len(pricing_with_discount[pricing_with_discount['active_sku_disc_pct'] == 0])}")
print(f"\nActive SKU discount distribution:")
print(pricing_with_discount[pricing_with_discount['active_sku_disc_pct'] > 0]['active_sku_disc_pct'].describe())
print(f"\nSample data with active SKU discounts:")
pricing_with_discount[pricing_with_discount['active_sku_disc_pct'] > 0][
    ['product_id', 'warehouse_id', 'sku', 'current_price', 'active_sku_disc_pct', 'discount_perc']
].head(15)


Loading active SKU discount data...
Loaded 11795 active SKU discount records

ACTIVE SKU DISCOUNT ANALYSIS
Records with active SKU discount: 15206
Records without active SKU discount: 93268

Active SKU discount distribution:
count    15206.000000
mean         1.958050
std          1.395844
min          0.250000
25%          0.865806
50%          1.583193
75%          2.810000
max          5.000000
Name: active_sku_disc_pct, dtype: float64

Sample data with active SKU discounts:


/tmp/ipykernel_18611/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,product_id,warehouse_id,sku,current_price,active_sku_disc_pct,discount_perc
2,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,240.00,1.161774,0.000000
3,10594,1,كورونا شوكولاتة داكنة 72%- 45 جنية,240.00,1.161774,0.000000
6,2424,337,حفاضات جود كير مقاس 5 - 40 حفاضة,178.25,4.350000,0.003046
8,9153,703,زيت شعر فيانسيه مغذى باللوز - 175 مل,30.00,1.670000,0.000000
12,4076,170,شوكولاتة جلاكسى سادة - 18 جم,157.00,0.870000,0.000000
17,25718,632,صلصة رشيدي الميزان برطمان - 300 جم,300.75,4.410000,0.000000
20,9609,337,سمن جنة - 2.5 كجم,1095.00,0.860000,0.000000
22,11497,1,مكرونة الملكة ملوكى خواتم - 400 جم,259.75,0.713385,0.000000
23,11497,1,مكرونة الملكة ملوكى خواتم - 400 جم,259.75,0.713385,0.000000
24,958,339,لبن المراعى تريتس موز - 200 مل,281.25,2.640000,0.005337


In [52]:
final_df = pricing_with_discount[(pricing_with_discount['no_nmv_4m']==0)|(pricing_with_discount['stocks']>0)]

In [ ]:
# Drop duplicates before saving
final_df = final_df.drop_duplicates(subset=['product_id', 'warehouse_id'], keep='first')
print(f"Exported {len(final_df)} records (duplicates removed)")

In [ ]:
final_df['created_at'] = TODAY
final_df['created_at'] =pd.to_datetime(final_df['created_at']).dt.date

In [ ]:
status = upload_dataframe_to_snowflake("Egypt", final_df, "MATERIALIZED_VIEWS", "Pricing_data_extraction", "append", auto_create_table=True, conn=None)

# Send Slack notification
if status:
    slack_message = f"""✅ *Data Extraction Script Completed Successfully*
    
📅 Date: {TODAY}
📊 Records uploaded: {len(final_df):,}
🗄️ Table: MATERIALIZED_VIEWS.Pricing_data_extraction
⏰ Completed at: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time"""
    
    send_text_slack('new-pricing-logic',slack_message)
    print("✅ Slack notification sent!")
else:
    error_message = f"""❌ *Data Extraction Script Failed*
    
📅 Date: {TODAY}
⏰ Failed at: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time
⚠️ Upload to Snowflake failed - please check logs"""
    
    send_text_slack('new-pricing-logic',error_message)
    print("❌ Error notification sent to Slack!")